# title: "SIMPlex Breast Cancer — spatial integration and deconvolution benchmarking"
subtitle: "Manuscript figures: Fig. 1g, Fig. 2a–h, Extended Data Fig. 4–6"
author: "Marcos Tito Machado"
output: html_document



In [ ]:
# DATA_ROOT, HDF5 library, palettes — single source of truth for paths.
source(here::here("config.R"))

In [ ]:
# HDF5 library is loaded by config.R
library(semla)
library(tibble)
library(Seurat)
options(Seurat.object.assay.version = "v5")
library(patchwork)
library(plotly)
library(singlet)
library(RcppML)
library(ggplot2)
library(dplyr)
library(tidyr)
library(viridis)
library(pheatmap)
library(cowplot)
library(corrplot)
library(RColorBrewer)
library(heatmap3)
library(harmony)
library(paletteer)
options(browser = "false") # to prevent an error due to remote browser



### Settings 


In [ ]:
rootObj <- paste0(SN_RDS, "/breast_cancer/cross_patient/")
wd <- paste0(FIGS_ROOT, "/breast_cancer/analysis_majorLevel/")
if (!dir.exists(rootObj)) {
    dir.create(rootObj, recursive = TRUE)
}
if (!dir.exists(wd)) {
    dir.create(wd, recursive = TRUE)
}

# COLORS%
colsCelltype <- rev(paletteer_c("grDevices::Temps", 9))
colsCelltype <- c("#EAE29CFF", "#6CC382FF", "#E99F69FF", "#CF597EFF", "#EAC17AFF", "#29AD8EFF", "#B2D387FF","#E4796DFF", "#089392FF")
colsSample <- paletteer_c("grDevices::Dark 3", 10)
colsSubtype <- c("#6EA9B0", "#B1A6D1", "#E18A96") # rev(paletteer_c("ggthemes::Red-Green-Gold Diverging", 3))
colsMalignancy <- c("#84D7E1","#6ABE8C","#FF95A8")
colsAssays <- c("#6A5ACD", "#FFA500", "#20B2AA")

format_k <- function(x) {
  ifelse(abs(x) >= 1000, paste0(round(x / 1000), "k"), as.character(x))
}



# Compile all simplex data so far

### Merge datasets


In [ ]:
data_path <- paste0(SN_RDS, "/breast_cancer/per_sample")
patient_csv <- read.csv(here::here("resources/sample_metadata.csv"))
included_samples <- c(
    "patient1_55um", 
    "patient2_55um", 
    "patient4_55um", 
    "patient4_HD", 
    "patient5_55um", 
    "patient5_HD", 
    "patient6_55um", 
    "patient7_55um", 
    "patient8_55um", 
    "patient9_55um"
)

spl_paths <- list()
for (sample in included_samples) {
    patient_info <- patient_csv[patient_csv$sample == sample, ]
    spl_paths[[sample]] <- list(
        path = file.path(data_path, paste0(sample, "/", sample, "_seuratAnnotation.rds")),
        patient_ID = patient_info$patient_ID,
        subtype = patient_info$subtype
    )
}

spls <- list()
for (splname in names(spl_paths)) {
    spl <- readRDS(spl_paths[[splname]]$path)
    spl@meta.data$sample <- splname
    spl@meta.data$patient_ID <- spl_paths[[splname]]$patient_ID
    spl@meta.data$subtype <- spl_paths[[splname]]$subtype
    spls[[splname]] <- spl
}
merged <- merge(x = spls[[1]], y = spls[-1])
table(merged@meta.data$sample)



### QC 


In [ ]:
qc_medians <- merged@meta.data %>%
    group_by(sample) %>%
    summarise(
        nFeature_RNA = median(nFeature_RNA, na.rm = TRUE),
        nCount_RNA = median(nCount_RNA, na.rm = TRUE)
    ) %>%
    tidyr::pivot_longer(
        cols = c(nFeature_RNA, nCount_RNA),
        names_to = "feature",
        values_to = "median_value"
    )

library(patchwork)
plots <- lapply(unique(qc_medians$feature), function(feat) {
    medians <- qc_medians %>% filter(feature == feat)
    # Get max value per sample for y position
    max_vals <- merged@meta.data %>%
        group_by(sample) %>%
        summarise(max_y = max(.data[[feat]], na.rm = TRUE))
    medians <- left_join(medians, max_vals, by = "sample")
    # Calculate x positions: right of each violin
    sample_levels <- levels(factor(merged@meta.data$sample))
    n_samples <- length(sample_levels)
    x_positions <- seq_along(sample_levels)  # shift right
    medians$x_pos <- x_positions[match(medians$sample, sample_levels)]
    # Set y axis limit to 1.2x the max value
    y_max <- max(medians$max_y, na.rm = TRUE) * 1.2
    p <- VlnPlot(
        merged,
        features = feat,
        group.by = "sample",
        pt.size = 0
    ) +
    geom_text(
        data = medians,
        aes(x = x_pos, y = max_y * 1.05, label = sprintf("%.0f", median_value)),
        size = 3,
        color = "black",
        inherit.aes = FALSE
    ) +
    ggtitle(feat) +
    theme(legend.position = "none") +
    ylim(0, y_max)
    return(p)
})

p <- wrap_plots(plots, ncol = 2)

ggsave(
    filename = paste0(FIGS_ROOT, "/breast_cancer/analysis_majorLevel/QC_metrics_postThreshold.pdf"), # modify accordingly
    plot = p,
    width = 8,
    height = 5
)



#### Filter
Eliminate nuclei with less than 200 genes


In [ ]:
merged <- subset(merged, subset = nFeature_RNA > 200)



#### Integrate all 


In [ ]:
wd <- paste0(FIGS_ROOT, "/breast_cancer/analysis_majorLevel/")
set.seed(1)
merged <- merged %>%
  NormalizeData() %>%
  FindVariableFeatures() %>%
  ScaleData() %>%
  RunPCA() %>%
  RunUMAP(dims = 1:30)
# plot pre integration
ggsave(
    filename = paste0(wd, "integration/pca_pre_integration.pdf"),
    plot = DimPlot(merged, reduction = "pca", group.by = "sample"),
    width = 10,
    height = 10
)
ggsave(
    filename = paste0(wd, "integration/umap_pre_integration.pdf"),
    plot = DimPlot(merged, reduction = "umap", group.by = c("sample", "seuratAnnotation_major", "subtype"), ncol = 3),
    width = 30,
    height = 10
)
# integrate
int.all <- merged %>% 
    RunHarmony("sample", plot_convergence = TRUE, theta = 0) # min thetha since we expect a lot of biological variation
int.all <- RunUMAP(int.all, dims = 1:30, reduction = "harmony")

# plot post integration
ggsave(
    filename = paste0(wd, "integration/pca_post_integration.pdf"),
    plot = DimPlot(int.all, reduction = "harmony", group.by = "sample"),
    width = 10,
    height = 10
)
ggsave(
    filename = paste0(wd, "integration/umap_post_integration.pdf"),
    plot = DimPlot(int.all, reduction = "umap", group.by = c("sample", "seuratAnnotation_major", "subtype"), ncol = 3),
    width = 30,
    height = 10
)



#### plot consecutive samples only


In [ ]:
p4 <- subset(int.all, subset = sample %in% c("patient4_55um", "patient4_HD"))
ggsave(
    filename = paste0(wd, "integration/patient4_umap_post_integration.pdf"),
    plot = DimPlot(p4, reduction = "umap", group.by = c("sample", "seuratAnnotation_major"), ncol = 3),
    width = 30,
    height = 10
)
p5 <- subset(int.all, subset = sample %in% c("patient5_55um", "patient5_HD"))
ggsave(
    filename = paste0(wd, "integration/patient5_umap_post_integration.pdf"),
    plot = DimPlot(p5, reduction = "umap", group.by = c("sample", "seuratAnnotation_major"), ncol = 3),
    width = 30,
    height = 10
)



#### Plot Cancer VS Healthy Epithelial 


In [ ]:
int.all@meta.data$malignancy <- ifelse(int.all@meta.data$predicted.id == "Cancer Epithelial", "Tumour Epithelial", "Benign")
int.all@meta.data$malignancy <- ifelse(
    int.all@meta.data$malignancy == "Benign" & int.all@meta.data$seuratAnnotation_major == "Epithelial",
    "Benign Epithelial",
    int.all@meta.data$malignancy
)
ggsave(
    filename = paste0(wd, "integration/cancer_vs_healthy_epi_umap.pdf"),
    plot = DimPlot(int.all, reduction = "umap", group.by = c("sample", "malignancy"), ncol = 2),
    width = 20,
    height = 10
)



## save 


In [ ]:
saveRDS(int.all, file = paste0(rootObj, "SIMPlex_breast_allSamples.rds"))



## Cell type representation versus bc_atlas Atlas / others

#### load others 


In [ ]:
bc_atlas <- readRDS(paste0(EXT_REFS, "/BC_atlas/miniatlas.rds"))
# merge epithelial layers
bc_atlas@meta.data$celltype_major <- ifelse(
    bc_atlas@meta.data$celltype_major %in% c("Normal Epithelial", "Cancer Epithelial"),
    "Epithelial",
    bc_atlas@meta.data$celltype_major
)

In [ ]:
int_counts <- table(int.all$seuratAnnotation_major)
bc_atlas_counts <- table(bc_atlas$celltype_major)

data <- data.frame(
  CellType = c(names(int_counts), names(bc_atlas_counts)),
  Count = c(as.numeric(int_counts), as.numeric(bc_atlas_counts)),
  Dataset = rep(c("SIMPlex", "BC Atlas"), times = c(length(int_counts), length(bc_atlas_counts))),
  N = rep(c(length(unique(int.all$sample)), length(unique(bc_atlas$Patient))), 
            times = c(length(int_counts), length(bc_atlas_counts)))
)

data <- data %>%
  group_by(Dataset) %>%
  mutate(Proportion = Count / sum(Count))

# COLORS
cols <- colsCelltype

# Create the stacked bar plot
p1 <- ggplot(data) + 
    aes(x = Dataset, y = Proportion, fill = CellType) +
    geom_bar(stat = "identity", position = "stack") +
    ylab("Celltype Proportions") +
    geom_text(aes(label = paste0(format_k(Count), " ", round(Proportion * 100, 1), "%")), 
              position = position_stack(vjust = 0.5), size = 3) +
    ggtitle("Cell Type Proportions") +
    theme_bw() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) + 
    scale_fill_manual(values = cols) +  
    annotate("text", x = 1, y = 1.05, label = paste0("N=", length(unique(bc_atlas$Patient)), "\nNcells=", length(bc_atlas$Patient)), size = 3) +
    annotate("text", x = 2, y = 1.05, label = paste0("N=", length(unique(int.all$sample)), "\nNnuclei=", length(int.all$sample)), size = 3)
            
ggsave(
    filename = paste0(wd, "final_plots/celltype_proportions_vs_miniatlas_stacked.pdf"),
    plot = p1,
    width = 4,
    height = 7
)

# COLORS
cols <- rev(paletteer_c("grDevices::Temps", 2))
p2 <- ggplot(data) + 
    aes(x = CellType, y = Proportion, fill = Dataset) +
    geom_bar(stat = "identity", position = "dodge") +
    ylab("Celltype Proportions") +
    geom_text(aes(label = Count), 
                        position = position_dodge(width = 0.9), size = 3, vjust = -0.5) +
    ggtitle("") +
    theme_bw() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) + 
    scale_fill_manual(values = cols)


ggsave(
    filename = paste0(wd, "final_plots/celltype_proportions_vs_miniatlas_dodge.pdf"),
    plot = p2,
    width = 10,
    height = 5
)

#### UMAP with same colors 
cols <- c("#EAE29CFF", "#6CC382FF", "#E99F69FF", "#CF597EFF", "#EAC17AFF", "#29AD8EFF", "#B2D387FF","#E4796DFF", "#089392FF")
ggsave(
    filename = paste0(wd, "umap_celltypes_recolored.pdf"),
    plot = DimPlot(int.all, reduction = "umap", group.by = "seuratAnnotation_major", cols = cols, label = TRUE),
    width = 6,
    height = 5
)



#### Sample, Subtype, Malignancy proportions


In [ ]:
dataname <- "malignancy"
int_data <- table(int.all$malignancy)
cols <- colsMalignancy

data <- data.frame(
    Category = dataname,
    Group = names(int_data),
    Nuclei = as.numeric(int_data),
    Proportion = as.numeric(int_data) / sum(as.numeric(int_data))
)

# Create the bar plot
p1 <- ggplot(data) + 
    aes(x = Category, y = Proportion, fill = Group) +
    geom_bar(stat = "identity", position = "stack") +
    ylab("Proportions") +
    geom_text(aes(label = Nuclei), 
              position = position_stack(vjust = 0.5), size = 3) +
    ggtitle("") +
    theme_bw() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) +  
    annotate("text", x = 1, y = 1.05, label = paste0("N=", length(unique(int.all$sample)), "\nNnuclei=", sum(as.numeric(int_data))), size = 3) + 
    scale_fill_manual(values = cols)

#### UMAP with same colors 
# Combine the bar plot and UMAP plot using patchwork
umap_plot <- DimPlot(int.all, reduction = "umap", group.by = dataname, label = FALSE, alpha = 0.2, pt.size = 0.001, cols = cols)
combined_plot <- p1 + umap_plot + plot_layout(widths = c(0.2, 0.8))

ggsave(
        filename = paste0(wd, dataname, "_proportions_umap.pdf"),
        plot = combined_plot,
        width = 9,
        height = 6
)



#### Benign epithelial patient distribution 



In [ ]:
benign_epithelial <- subset(int.all, subset = malignancy == "Benign Epithelial")
dataname <- "sample"
int_data <- table(benign_epithelial$sample)
cols <- colsSample

data <- data.frame(
    Category = dataname,
    Group = names(int_data),
    Nuclei = as.numeric(int_data),
    Proportion = as.numeric(int_data) / sum(as.numeric(int_data))
)

# Create the bar plot
p1 <- ggplot(data) + 
    aes(x = Category, y = Proportion, fill = Group) +
    geom_bar(stat = "identity", position = "stack") +
    ylab("Proportions") +
    geom_text(aes(label = Nuclei), 
              position = position_stack(vjust = 0.5), size = 3) +
    ggtitle("Samples across benign epithelial") +
    theme_bw() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) +  
    annotate("text", x = 1, y = 1.05, label = paste0("N=", length(unique(int.all$sample)), "\nNnuclei=", sum(as.numeric(int_data))), size = 3) + 
    scale_fill_manual(values = cols)

ggsave(
        filename = paste0(wd, "final_plots/", dataname, "_benignEpithelial.pdf"),
        plot = p1,
        width = 3.5,
        height = 6
)



## Deconvolution of major cell types vs miniatlas

#### load BC Atlas atlas


In [ ]:
bc_atlas <- readRDS(paste0(EXT_REFS, "/BC_atlas/miniatlas.rds"))
# merge epithelial layers
bc_atlas@meta.data$celltype_major <- ifelse(
    bc_atlas@meta.data$celltype_major %in% c("Normal Epithelial", "Cancer Epithelial"),
    "Epithelial",
    bc_atlas@meta.data$celltype_major
)



#### Map ALL SIMPlex to Spatial data


In [ ]:
samples <- unique(int.all$sample)
spls <- list()

# ref steps 
DefaultAssay(int.all) <- "RNA"
int.all <- int.all |>
FindVariableFeatures(nfeatures = 2000)
for (sample in samples){
    vis <- readRDS(paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/", sample, "_spatial.rds"))

    ## Map w/ semla NNLS
    ident <- "seuratAnnotation_major" # ident in the reference to use for mapping 
    celltype <- "simplex_major_map" # name of the mapping to assign to the visium object
    DefaultAssay(vis) <- "Spatial"
    vis <- vis |>
    NormalizeData() |>
    FindVariableFeatures(nfeatures = nrow(vis)) |>
    ScaleData()
    vis <- RunNNLS(vis, singlecell_object = int.all, spatial_assay = DefaultAssay(vis), singlecell_assay = DefaultAssay(int.all), groups = ident, slot = "data", assay_name = celltype)
    DefaultAssay(vis) <- celltype
    spls[[sample]] <- vis
}

# Plot each sample separately
sample_plots <- list()
hd_sample_plots <- list()

for (sample in samples) {
    features = unique(gsub("_", "-", int.all$seuratAnnotation_major))
    vis <- spls[[sample]]
    if (length(vis$sample_id) > 5000) { # 11*11 Array Visium
        pt_size <- 1
    } else if (length(vis$sample_id) < 5000) { # 6.5*6.5 Array Visium
        pt_size <- 1.6
    } else if (grepl("HD", sample)){
        pt_size <- 0.1
    }
    if (grepl("HD", sample)) {
        p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = pt_size, override_plot_dims = TRUE, scale = "shared", ncol = length(features), shape = "tile")
        hd_sample_plots[[sample]] <- p # & scale_fill_viridis_c(limits = c(0,1), option = "H")
    } else {
        p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = pt_size, override_plot_dims = TRUE, scale = "shared", ncol = length(features))
        sample_plots[[sample]] <- p & scale_fill_viridis_c(limits = c(0,1), option = "H")
    }
}

# Merge non-HD sample plots into a final plot
final_plot <- patchwork::wrap_plots(sample_plots, ncol = 1)
ggsave(paste0(wd, "map/", "simplex_major_map.jpg"), final_plot, height = 8*5, width = 9*5, dpi = 300, limitsize = FALSE)

# Merge HD sample plots into a separate final plot
if (length(hd_sample_plots) > 0) {
    hd_final_plot <- patchwork::wrap_plots(hd_sample_plots, ncol = 1)
    ggsave(paste0(wd, "map/", "simplex_major_mapHD.jpg"), hd_final_plot, height = 4*5, width = 9*5, dpi = 300, limitsize = FALSE)
}



#### Map BC Atlas Atlas to Spatial data


In [ ]:
samples <- unique(int.all$sample)

# ref steps 
DefaultAssay(bc_atlas) <- "RNA"
bc_atlas <- bc_atlas |>
FindVariableFeatures(nfeatures = 2000)
for (sample in samples){
    vis <- spls[[sample]]

    ## Map w/ semla NNLS
    ident <- "celltype_major" # ident in the reference to use for mapping 
    celltype <- "garvan_major_map" # name of the mapping to assign to the visium object
    DefaultAssay(vis) <- "Spatial"
    # vis <- vis |> # this process was already done in the previous chunk
    # NormalizeData() |>
    # FindVariableFeatures(nfeatures = nrow(vis)) |>
    # ScaleData()
    vis <- RunNNLS(vis, singlecell_object = bc_atlas, spatial_assay = DefaultAssay(vis), singlecell_assay = DefaultAssay(bc_atlas), groups = ident, slot = "data", assay_name = celltype)
    DefaultAssay(vis) <- celltype
    spls[[sample]] <- vis
}

# Plot each sample separately
sample_plots <- list()
hd_sample_plots <- list()

for (sample in samples) {
    features = unique(gsub("_", "-", bc_atlas$celltype_major))
    vis <- spls[[sample]]
    if (length(vis$sample_id) > 5000) { # 11*11 Array Visium
        pt_size <- 1
    } else if (length(vis$sample_id) < 5000) { # 6.5*6.5 Array Visium
        pt_size <- 1.6
    } else if (grepl("HD", sample)){
        pt_size <- 0.1
    }
    if (grepl("HD", sample)) {
        p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = pt_size, override_plot_dims = TRUE, scale = "shared", ncol = length(features), shape = "tile")
        hd_sample_plots[[sample]] <- p # & scale_fill_viridis_c(limits = c(0,1), option = "H")
    } else {
        p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = pt_size, override_plot_dims = TRUE, scale = "shared", ncol = length(features))
        sample_plots[[sample]] <- p & scale_fill_viridis_c(limits = c(0,1), option = "H")
    }
}

# Merge non-HD sample plots into a final plot
final_plot <- patchwork::wrap_plots(sample_plots, ncol = 1)
ggsave(paste0(wd, "map/", "garvan_major_map.jpg"), final_plot, height = 8*5, width = 9*5, dpi = 300, limitsize = FALSE)

# Merge HD sample plots into a separate final plot
if (length(hd_sample_plots) > 0) {
    hd_final_plot <- patchwork::wrap_plots(hd_sample_plots, ncol = 1)
    ggsave(paste0(wd, "map/", "garvan_major_mapHD.jpg"), hd_final_plot, height = 4*5, width = 9*5, dpi = 300, limitsize = FALSE)
}



#### Patient specific mapping


In [ ]:
samples <- unique(int.all$sample)

for (sample in samples){
    vis <- spls[[sample]]
    patient <- sub("_.*", "", sample)
    ## patspec: patient specific mapping 
    DefaultAssay(int.all) <- "RNA"
    patspec <- subset(int.all, subset = patient_ID == !!patient)
    DefaultAssay(patspec) <- "RNA"
    patspec <- patspec |>
    NormalizeData() |>
    FindVariableFeatures(nfeatures = 2000) |>
    ScaleData()

    ## Map w/ semla NNLS
    ident <- "seuratAnnotation_major" # ident in the reference to use for mapping 
    celltype <- "patspec_major_map" # name of the mapping to assign to the visium object
    DefaultAssay(vis) <- "Spatial"
    # vis <- vis |> # this process was already done in the previous chunk
    # NormalizeData() |>
    # FindVariableFeatures(nfeatures = nrow(vis)) |>
    # ScaleData()
    vis <- RunNNLS(vis, singlecell_object = patspec, spatial_assay = DefaultAssay(vis), singlecell_assay = DefaultAssay(patspec), groups = ident, slot = "data", assay_name = celltype)
    DefaultAssay(vis) <- celltype
    spls[[sample]] <- vis
}

# Plot each sample separately
sample_plots <- list()
hd_sample_plots <- list()

for (sample in samples) {
    vis <- spls[[sample]]
    features = rownames(vis@assays$patspec_major_map)
    if (length(vis$sample_id) > 5000) { # 11*11 Array Visium
        pt_size <- 1
    } else if (length(vis$sample_id) < 5000) { # 6.5*6.5 Array Visium
        pt_size <- 1.6
    } else if (grepl("HD", sample)){
        pt_size <- 0.1
    }
    if (grepl("HD", sample)) {
        p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = pt_size, override_plot_dims = TRUE, scale = "shared", ncol = length(features), shape = "tile")
        hd_sample_plots[[sample]] <- p & scale_fill_viridis_c(limits = c(0,1), option = "H")
    } else {
        p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = pt_size, override_plot_dims = TRUE, scale = "shared", ncol = length(features))
        sample_plots[[sample]] <- p & scale_fill_viridis_c(limits = c(0,1), option = "H")
    }
}

# Merge non-HD sample plots into a final plot
final_plot <- patchwork::wrap_plots(sample_plots, ncol = 1)
ggsave(paste0(wd, "map/", "patspec_major_map.jpg"), final_plot, height = 8*5, width = 9*5, dpi = 300, limitsize = FALSE)

# Merge HD sample plots into a separate final plot
if (length(hd_sample_plots) > 0) {
    hd_final_plot <- patchwork::wrap_plots(hd_sample_plots, ncol = 1)
    ggsave(paste0(wd, "map/", "patspec_major_mapHD.jpg"), hd_final_plot, height = 4*5, width = 9*5, dpi = 300, limitsize = FALSE)
}



#### Control mapping


In [ ]:
samples <- unique(int.all$sample)

for (sample in samples){
    vis <- spls[[sample]]
    patient <- sub("_.*", "", sample)
    ## control: remove the sample itself from the reference
    DefaultAssay(int.all) <- "RNA"
    control <- subset(int.all, subset = patient_ID != !!patient)
    DefaultAssay(control) <- "RNA"
    control <- control |>
    NormalizeData() |>
    FindVariableFeatures(nfeatures = 2000) |>
    ScaleData()

    ## Map w/ semla NNLS
    ident <- "seuratAnnotation_major" # ident in the reference to use for mapping 
    celltype <- "control_major_map" # name of the mapping to assign to the visium object
    DefaultAssay(vis) <- "Spatial"
    # vis <- vis |> # this process was already done in the previous chunk
    # NormalizeData() |>
    # FindVariableFeatures(nfeatures = nrow(vis)) |>
    # ScaleData()
    vis <- RunNNLS(vis, singlecell_object = control, spatial_assay = DefaultAssay(vis), singlecell_assay = DefaultAssay(control), groups = ident, slot = "data", assay_name = celltype)
    DefaultAssay(vis) <- celltype
    spls[[sample]] <- vis
}

# Plot each sample separately
sample_plots <- list()
hd_sample_plots <- list()

for (sample in samples) {
    features = unique(gsub("_", "-", control$seuratAnnotation_major))
    vis <- spls[[sample]]
    if (length(vis$sample_id) > 5000) { # 11*11 Array Visium
        pt_size <- 1
    } else if (length(vis$sample_id) < 5000) { # 6.5*6.5 Array Visium
        pt_size <- 1.6
    } else if (grepl("HD", sample)){
        pt_size <- 0.1
    }
    if (grepl("HD", sample)) {
        p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = pt_size, override_plot_dims = TRUE, scale = "shared", ncol = length(features), shape = "tile")
        hd_sample_plots[[sample]] <- p # & scale_fill_viridis_c(limits = c(0,1), option = "H")
    } else {
        p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = pt_size, override_plot_dims = TRUE, scale = "shared", ncol = length(features))
        sample_plots[[sample]] <- p & scale_fill_viridis_c(limits = c(0,1), option = "H")
    }
}

# Merge non-HD sample plots into a final plot
final_plot <- patchwork::wrap_plots(sample_plots, ncol = 1)
ggsave(paste0(wd, "map/", "control_major_map.jpg"), final_plot, height = 8*5, width = 9*5, dpi = 300, limitsize = FALSE)

# Merge HD sample plots into a separate final plot
if (length(hd_sample_plots) > 0) {
    hd_final_plot <- patchwork::wrap_plots(hd_sample_plots, ncol = 1)
    ggsave(paste0(wd, "map/", "control_major_mapHD.jpg"), hd_final_plot, height = 4*5, width = 9*5, dpi = 300, limitsize = FALSE)
}



#### Save visium objects
save 55um objects, but HD we save the decon as csv since the objects are too large 


In [ ]:
assays <- c("patspec_major_map", "control_major_map", "garvan_major_map", "simplex_major_map")
for (sample in samples) {
    output_path <- paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/")
    if (grepl("HD", sample)) {
        for (assay in assays) {
            DefaultAssay(spls[[sample]]) <- assay
            assay_data <- as.data.frame(t(as.matrix(spls[[sample]]@assays[[assay]]@data)))
            write.csv(assay_data, file = paste0(output_path, sample, "_", assay, ".csv"), row.names = TRUE)
        }
    } else {
        output_file <- paste0(output_path, sample, "_3decon.rds")
        saveRDS(spls[[sample]], file = output_file)
    }
}



###### load saved visium objects


In [ ]:
int.all <- readRDS(paste0(rootObj, "SIMPlex_breast_allSamples.rds"))
samples <- unique(int.all$sample[!grepl("HD", int.all$sample)])
spls <- list()
for (sample in samples) {
    if (!grepl("HD", sample)) {
        output_path <- paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/", sample, "_3decon.rds")
        if (file.exists(output_path)) {
            spls[[sample]] <- readRDS(output_path)
        }
    }
}



###### merge 


In [ ]:
spls <- spls[!grepl("HD", names(spls))]
assays <- c("patspec_major_map", "control_major_map", "garvan_major_map", "simplex_major_map")
for (sample in names(spls)) {
    for (assay in assays) {
        DefaultAssay(spls[[sample]]) <- assay
        spls[[sample]] <- FindVariableFeatures(spls[[sample]], nfeatures = 2000)
    }
}
mergedVis <- MergeSTData(spls[[1]], spls[-1])
for (assay in assays) {
    DefaultAssay(mergedVis) <- assay
    mergedVis[[assay]]@var.features <- unique(unlist(lapply(spls, function(x) x[[assay]]@var.features)))
}
table(mergedVis@meta.data$sample_id)



### Moran's I 
here we try to see how clustered is the data compared to miniatlas deconvolution


In [ ]:
assays <- c("patspec_major_map", "control_major_map", "garvan_major_map", "simplex_major_map")
morans <- list()
for (assay in assays) {
    DefaultAssay(mergedVis) <- assay
    moran_res <- CorSpatialFeatures(mergedVis, across_all = TRUE)
    morans[[assay]] <- moran_res
    avgMorans[assay] <- mean(moran_res$cor, na.rm = TRUE)
}



We observe that SIMPlex reaches higher morans correlation but visualization is hard since the numerical differences are not huge 
despite probabily meaningful. 

### 3D correlation of SIMPlex, BC Atlas Atlas and Control

#### get correlation list 


In [ ]:
assays <- c("patspec_major_map", "control_major_map", "garvan_major_map")
comparisons <- list(c(assays[1], assays[2]), c(assays[2], assays[3]), c(assays[1], assays[3]))

cors <- list()
for (comparison in comparisons){
    shared <- intersect(rownames(mergedVis[[comparison[1]]]), rownames(mergedVis[[comparison[2]]]))
    DefaultAssay(mergedVis) <- comparison[1]
    data1 <- FetchData(mergedVis, shared)
    DefaultAssay(mergedVis) <- comparison[2]
    data2 <- FetchData(mergedVis, shared)
    cors[[paste(comparison, collapse = "_")]] <- cor(data1, data2, method = "spearman")
}



#### correlation of each 


In [ ]:
for (name in names(cors)) {
    cor_matrix <- cors[[name]]
    cor_matrix_name <- name  # Use the name directly
    # Generate heatmap and save as PDF
    dir.create(paste0(wd, "celltype_correlation/"), recursive = TRUE, showWarnings = FALSE)
    pdf(paste0(wd, "celltype_correlation/", cor_matrix_name, ".pdf"), width = 11, height = 8)
    heatmap3(
        cor_matrix, 
        margins = c(10, 10), 
        symm = TRUE, 
        scale = "none"
    )
    dev.off()  # Close the PDF device
}



### correlation of decon to marker genes

#### load objects


In [ ]:
int.all <- readRDS(paste0(rootObj, "SIMPlex_breast_allSamples.rds"))
samples <- unique(int.all$sample[!grepl("HD", int.all$sample)])
spls <- list()
for (sample in samples) {
    if (!grepl("HD", sample)) {
        output_path <- paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/", sample, "_3decon.rds")
        if (file.exists(output_path)) {
            spls[[sample]] <- readRDS(output_path)
        }
    }
}
# merge
mergedVis <- MergeSTData(spls[[1]], spls[-1])



#### Known Marker genes 


In [ ]:
genesets <- list()
  genesets$epithelial_markers   = c("EPCAM", "KRT8", "KRT5", "KRT14", "CDH1", "MUC1", "CLDN3", "CLDN4", "CLDN7")
  genesets$tcell_markers        = c("CD3D", "CD3E", "CD3G", "CD2", "TRAC")
  genesets$bcell_markers        = c("CD79B", "MS4A1", "BANK1")
  genesets$myeloid_markers      = c("ITGAM", "MRC1", "CD68","FUT4", "FCGR3A")
  genesets$plasmablast_markers  = c("XBP1", "PRDM1", "JCHAIN", "IGHG1")
  genesets$fibroblast_markers   = c("COL1A1", "COL1A2", "DCN", "FAP", "MMP2", "MMP11", "THY1")
  genesets$endothelial_markers  = c("PECAM1", "VWF", "CDH5", "ESAM", "TIE1", "TEK", "FLT1", "KDR",
                           "CLDN5", "PLVAP", "PROX1", "LYVE1", "PDPN")
  genesets$pvl_markers          = c("MCAM", "ACTA2", "PDGFRB")



#### Ucell module score


In [ ]:
library(UCell)
# add UCell scores as metadata columns in mergedVis
mergedVis <- JoinLayers(mergedVis)
DefaultAssay(mergedVis) <- "Spatial"
mergedVis <- AddModuleScore_UCell(
  mergedVis,
  features = genesets
)



#### map marker modules


In [ ]:
modules <- grep("_UCell", colnames(mergedVis@meta.data), value = TRUE)
for (module in modules){
    message("Processing module: ", module)
    pw <- list()
    for (sample in samples){
        message("Processing sample: ", sample)
        vis <- SubsetSTData(mergedVis, expression = sample_id == !!sample)
        DefaultAssay(vis) <- "Spatial"
        # set point size based on number of spots
        pt_size <- if (length(vis$sample_id) > 5000) 1 else 1.7 
        pw[[sample]] <- MapFeatures(
            vis,
            features = module,
            colors = viridis::turbo(11),
            ncol = 1,
            label_by = "sample_id",
            override_plot_dims = TRUE,
            drop_na = TRUE,
            pt_size = pt_size
        )
    }
    pw <- patchwork::wrap_plots(pw, ncol = 1)
    ggsave(
        filename = paste0(wd, "map_marker_genes_modules/", module, ".jpg"),
        plot = pw,
        width = length(module) * 5,
        height = 5*8
    )
}



#### transform modules into assay 


In [ ]:
module_data <- mergedVis@meta.data[, modules]
module_matrix <- t(as.matrix(module_data))
module_assay <- CreateAssayObject(data = module_matrix)
mergedVis[["celltype_modules_UCell"]] <- module_assay



#### correlate marker modules to decon 


In [ ]:
#### get correlation list 
assays <- c("patspec_major_map", "control_major_map", "garvan_major_map")
cors <- list()
for (assay in assays){
    DefaultAssay(mergedVis) <- assay
    data1 <- FetchData(mergedVis, rownames(mergedVis))
    DefaultAssay(mergedVis) <- "celltype_modules_UCell"
    data2 <- FetchData(mergedVis, rownames(mergedVis))
    # Remove the suffix "-UCell" from column names in data2
    colnames(data2) <- gsub("-markers-UCell$", "", colnames(data2))
    # Exclude adipocytes
    data1 <- data1[, !grepl("Adipocyte", colnames(data1))]
    data2 <- data2[, !grepl("Adipocyte", colnames(data2))]
    # Custom order for data1 and data2
    custom_order_data1 <- c("B-cells", "T-cells", "Plasmablasts", "Myeloid", "Epithelial", "CAFs", "Endothelial", "PVL")
    custom_order_data2 <- c("bcell", "tcell", "plasmablast", "myeloid", "epithelial", "fibroblast", "endothelial", "pvl")
    
    # Reorder columns based on custom order
    data1 <- data1[, custom_order_data1, drop = FALSE]
    data2 <- data2[, custom_order_data2, drop = FALSE]
    cors[[paste(assay, collapse = "")]] <- cor(data1, data2, method = "spearman")
}

for (nm in names(cors)) {
    mat <- cors[[nm]]
    # colnames(mat) <- gsub("-markers$", "", colnames(mat))
    cors[[nm]] <- mat
}
for (name in names(cors)) {
    cor_matrix <- cors[[name]]
    cor_matrix_name <- name  # Use the name directly
    # Generate heatmap and save as PDF
    dir.create(paste0(wd, "celltype_correlation/"), recursive = TRUE, showWarnings = FALSE)
    pdf(paste0(wd, "celltype_correlation/UCell_modules_vs_", cor_matrix_name, ".pdf"), width = 10, height = 10)
    heatmap3(
        cor_matrix, 
        margins = c(12, 12), 
        symm = TRUE, 
        scale = "none",
        Rowv = NA, 
        Colv = NA, 
        na.rm = TRUE   
    )
    dev.off()  # Close the PDF device
}



##### Boxplot of marker-decon correlation



In [ ]:
correspondences <- list(
    "B-cells" = "bcell",
    "T-cells" = "tcell",
    "Plasmablasts" = "plasmablast",
    "Myeloid" = "myeloid",
    "Epithelial" = "epithelial",
    "CAFs" = "fibroblast",
    "Endothelial" = "endothelial",
    "PVL" = "pvl"
)

correlation_values <- list()
for (assay in names(cors)) {
    cor_matrix <- cors[[assay]]
    assay_correlations <- sapply(names(correspondences), function(celltype) {
        modules <- correspondences[[celltype]]
        if (celltype %in% rownames(cor_matrix)) {
            valid_modules <- modules[modules %in% colnames(cor_matrix)]
            if (length(valid_modules) > 0) {
                return(mean(cor_matrix[celltype, valid_modules], na.rm = TRUE))
            }
        }
        return(NA)
    })
    correlation_values[[assay]] <- assay_correlations
}

total_correlation_scores <- sapply(correlation_values, function(assay_correlations) {
    mean(assay_correlations, na.rm = TRUE)
})

# Create a data frame for visualization
correlation_df <- do.call(rbind, lapply(names(correlation_values), function(assay) {
    data.frame(
        Assay = assay,
        CellType = names(correlation_values[[assay]]),
        Correlation = correlation_values[[assay]]
    )
}))

# Add mean correlation for each assay
mean_correlation_df <- data.frame(
    Assay = names(total_correlation_scores),
    CellType = "Mean",
    Correlation = total_correlation_scores
)

# Combine both data frames
mean_correlation_df$Assay <- recode(mean_correlation_df$Assay, 
                                    "control_major_map" = "Neg.\n Control", 
                                    "garvan_major_map" = "Ref.\n Atlas", 
                                    "patspec_major_map" = "SIMPlex")
correlation_df$Assay <- recode(correlation_df$Assay, 
                               "control_major_map" = "Neg.\n Control", 
                               "garvan_major_map" = "Ref.\n Atlas", 
                               "patspec_major_map" = "SIMPlex")
correlation_df <- rbind(correlation_df, mean_correlation_df)

# Plot
plot <- ggplot(correlation_df, aes(x = Assay, y = Correlation)) +
    geom_boxplot(data = subset(correlation_df, CellType != "Mean"),
                 aes(fill = Assay), alpha = 0.25, color = "black", outlier.shape = NA) +  # Boxplot for distribution
    geom_jitter(data = subset(correlation_df, CellType != "Mean"),
                aes(color = CellType), width = 0.2, height = 0, size = 2) +  # Cell types
    ggrepel::geom_text_repel(data = subset(correlation_df, CellType != "Mean" & 
                                           (Correlation > quantile(Correlation, 0) | 
                                            Correlation < quantile(Correlation, 1))),
                             aes(label = CellType, color = CellType), 
                             size = 2, 
                             box.padding = 0.2, 
                             point.padding = 0.1, 
                             max.overlaps = 5) +  # Annotate outlier cell types with repelling text
    scale_color_manual(values = colsCelltype[-1]) +  # Use custom colors, ignoring the first color
    scale_fill_manual(values = colsAssays) +  # Use custom colors for Assay
    labs(title = "Deconv. correlation to marker genes",
         x = "Assay", y = "Spearman Correlation Coeff.") +
    theme_minimal() +
    theme(legend.title = element_blank(),
          panel.grid = element_blank(),
          axis.line = element_line(color = "black"),  # Add axis lines
          axis.ticks = element_line(color = "black"))  # Add axis ticks

ggsave(
    filename = paste0(wd, "celltype_correlation/UCell_boxplot_modules_vs_decon.pdf"),
    plot = plot,
    width = 4,
    height = 4
)



#### per-sample cell-module correlation



In [ ]:
correspondences <- list(
    "B-cells" = "bcell",
    "T-cells" = "tcell",
    "Plasmablasts" = "plasmablast",
    "Myeloid" = "myeloid",
    "Epithelial" = "epithelial",
    "CAFs" = "fibroblast",
    "Endothelial" = "endothelial",
    "PVL" = "pvl"
)
assays    <- c("control_major_map", "garvan_major_map", "patspec_major_map")
celltypes <- names(correspondences)
modules   <- unname(unlist(correspondences))

cor_list <- list()

for (assay in assays) {
  message("Processing assay: ", assay)
  
  # deconvolution scores for this assay
  DefaultAssay(mergedVis) <- assay
  deconv_mat <- FetchData(mergedVis, vars = rownames(mergedVis))
  
  # UCell module scores
  DefaultAssay(mergedVis) <- "celltype_modules_UCell"
  module_mat <- FetchData(mergedVis, vars = rownames(mergedVis))
  colnames(module_mat) <- gsub("-markers-UCell$", "", colnames(module_mat))
  
  # patients
  sample_ids   <- mergedVis$sample_id
  uniq_samples <- unique(sample_ids)
  
  for (sid in uniq_samples) {
    idx <- which(sample_ids == sid)
    
    d_s <- deconv_mat[idx, , drop = FALSE]
    m_s <- module_mat[idx, , drop = FALSE]
    
    # 8x8 Spearman correlation matrix (celltypes x modules)
    cor_mat <- suppressWarnings(
      cor(d_s, m_s, method = "spearman", use = "pairwise.complete.obs")
    )
    
    # pick the matching celltype ↔ module correlation (diagonal defined by 'correspondences')
    vals <- sapply(celltypes, function(ct) {
      mod <- correspondences[[ct]]
      if (ct %in% rownames(cor_mat) && mod %in% colnames(cor_mat)) {
        cor_mat[ct, mod]
      } else {
        NA_real_
      }
    })
    
    cor_list[[length(cor_list) + 1]] <- data.frame(
      sample_id   = sid,
      Assay       = assay,
      CellType    = celltypes,
      Correlation = as.numeric(vals)
    )
  }
}

correlation_df <- bind_rows(cor_list)

# Quick sanity check
table(correlation_df$Assay, correlation_df$CellType)
head(correlation_df)



#### wilcoxon paired test between assays


In [ ]:
pairwise_results <- correlation_df %>%
  pivot_wider(names_from = Assay, values_from = Correlation) %>%
  group_by(CellType) %>%
  summarise(
    p_SIMPlex_vs_Ref  = wilcox.test(patspec_major_map, garvan_major_map,
                                    paired = TRUE)$p.value,
    p_SIMPlex_vs_Ctrl = wilcox.test(patspec_major_map, control_major_map,
                                    paired = TRUE)$p.value,
    p_Ref_vs_Ctrl     = wilcox.test(garvan_major_map, control_major_map,
                                    paired = TRUE)$p.value,
    .groups = "drop"
  )

pairwise_results

pairwise_results_BH <- pairwise_results %>%
  mutate(
    p_SIMPlex_vs_Ref_BH  = p.adjust(p_SIMPlex_vs_Ref,  method = "BH"),
    p_SIMPlex_vs_Ctrl_BH = p.adjust(p_SIMPlex_vs_Ctrl, method = "BH"),  
    p_Ref_vs_Ctrl_BH     = p.adjust(p_Ref_vs_Ctrl,     method = "BH")
  )

pairwise_results_BH



#### plot with included statistics 


In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)

## 1. Assay factor (same as you had)
corr_plot_df <- correlation_df %>%
  mutate(
    Assay = factor(
      Assay,
      levels = c("control_major_map", "garvan_major_map", "patspec_major_map"),
      labels = c("Neg. \nControl", "Ref. \nAtlas", "SIMPlex")
    )
  )

assay_levels <- levels(corr_plot_df$Assay)

## 2. Build long table of pairwise BH-corrected p values
sig_long <- pairwise_results_BH %>%
  transmute(
    CellType,
    `SIMPlex vs Ref`  = p_SIMPlex_vs_Ref_BH,
    `SIMPlex vs Ctrl` = p_SIMPlex_vs_Ctrl_BH,
    `Ref vs Ctrl`     = p_Ref_vs_Ctrl_BH
  ) %>%
  pivot_longer(
    cols      = -CellType,
    names_to  = "comparison",
    values_to = "q"
  ) %>%
  mutate(
    label = case_when(
      q < 0.001 ~ "***",
      q < 0.01  ~ "**",
      q < 0.05  ~ "*",
      q < 0.10  ~ "·",   # trend
      TRUE     ~ "ns"
    ),
    group1 = case_when(
      comparison == "SIMPlex vs Ref"  ~ "SIMPlex",
      comparison == "SIMPlex vs Ctrl" ~ "SIMPlex",
      comparison == "Ref vs Ctrl"     ~ "Ref. \nAtlas"
    ),
    group2 = case_when(
      comparison == "SIMPlex vs Ref"  ~ "Ref. \nAtlas",
      comparison == "SIMPlex vs Ctrl" ~ "Neg. \nControl",
      comparison == "Ref vs Ctrl"     ~ "Neg. \nControl"
    )
  )

## 3. Base y position per CellType (top of panel)
y_base <- corr_plot_df %>%
  group_by(CellType) %>%
  summarise(y0 = max(Correlation, na.rm = TRUE) + 0.03, .groups = "drop")

## 4. Add vertical offsets so brackets don’t overlap
sig_long <- sig_long %>%
  left_join(y_base, by = "CellType") %>%
  mutate(
    y = case_when(
      comparison == "SIMPlex vs Ref"  ~ y0 + 0.00,
      comparison == "SIMPlex vs Ctrl" ~ y0 + 0.07,
      comparison == "Ref vs Ctrl"     ~ y0 + 0.14
    ),
    x1 = as.numeric(factor(group1, levels = assay_levels)),
    x2 = as.numeric(factor(group2, levels = assay_levels)),
    x_mid = (x1 + x2) / 2
  ) %>%
  # drop non-significant if you don’t want "ns" clutter:
  filter(label != "ns")

## 5. Plot with all pairwise comparisons shown
p <- ggplot(corr_plot_df, aes(x = Assay, y = Correlation)) +
  geom_boxplot(
    aes(fill = Assay),
    alpha = 0.4,
    outlier.shape = NA,
    width = 0.6,
    color = "black"
  ) +
  geom_jitter(
    aes(group = Assay),
    width = 0.15,
    size  = 1.6,
    alpha = 0.7
  ) +
  facet_wrap(~ CellType, ncol = 4) +
  # significance brackets
  geom_segment(
    data = sig_long,
    aes(x = x1, xend = x2, y = y, yend = y),
    inherit.aes = FALSE,
    linewidth = 0.4
  ) +
  geom_text(
    data = sig_long,
    aes(x = x_mid, y = y + 0.01, label = label),
    inherit.aes = FALSE,
    size = 5
  ) +
  scale_fill_manual(
    values = c(
      "Neg. \nControl" = colsAssays[1],
      "Ref. \nAtlas"   = colsAssays[2],
      "SIMPlex"        = colsAssays[3]
    )
  ) +
  labs(
    title = "Per-patient deconvolution vs marker correlation",
    x     = "Deconvolution reference",
    y     = "Spearman Correlation Coefficient"
  ) +
  theme_bw(base_size = 11) +
  theme(
    legend.position  = "bottom",
    legend.title     = element_blank(),
    strip.background = element_rect(fill = "grey90", colour = NA),
    panel.grid       = element_blank()
  )

p

ggsave(
  filename = file.path(wd, "celltype_correlation/UCell_boxplot_statistics_all_pairs.pdf"),
  plot = p,
  width = 7.5,
  height = 5
)



#### save progress


In [ ]:
saveRDS(mergedVis, file = paste0(rootObj, "/VISIUM_breast_allSamples.rds"))
for (sample in samples) {
    output_path <- paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/", sample, "_3decon.rds")
    saveRDS(spls[[sample]], file = output_path)
}



#### load progress


In [ ]:
int.all <- readRDS(paste0(rootObj, "SIMPlex_breast_allSamples.rds"))
# mergedVis <- readRDS(paste0(rootObj, "/VISIUM_breast_allSamples.rds"))
samples <- unique(int.all$sample[!grepl("HD", int.all$sample)])
spls <- list()
for (sample in samples) {
    output_path <- paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/", sample, "_3decon.rds")
    if (file.exists(output_path)) {
        spls[[sample]] <- readRDS(output_path)
    }
}



## technical deconvolution of patient labels



In [ ]:
samples <- unique(int.all$sample[!grepl("HD", int.all$sample)])
for (sample in samples){
    vis <- readRDS(paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/", sample, "_spatial.rds"))
    DefaultAssay(int.all) <- "RNA"

    ## Map w/ semla NNLS
    ident <- "sample" # ident in the reference to use for mapping 
    celltype <- "sampleID_map" # name of the mapping to assign to the visium object
    DefaultAssay(vis) <- "Spatial"
    vis <- vis |>
    NormalizeData() |>
    FindVariableFeatures(nfeatures = nrow(vis)) |>
    ScaleData()
    vis <- RunNNLS(vis, singlecell_object = int.all, spatial_assay = DefaultAssay(vis), singlecell_assay = DefaultAssay(int.all), groups = ident, slot = "data", assay_name = celltype)
    spls[[sample]]@assays[[celltype]] <- vis@assays[[celltype]]
    DefaultAssay(spls[[sample]]) <- celltype
}

# Plot each sample separately
sample_plots <- list()
hd_sample_plots <- list()

for (sample in samples) {
    vis <- spls[[sample]]
    features = rownames(vis@assays$sampleID_map)
    if (length(vis$sample_id) > 5000) { # 11*11 Array Visium
        pt_size <- 1
    } else if (length(vis$sample_id) < 5000) { # 6.5*6.5 Array Visium
        pt_size <- 1.6
    } else if (grepl("HD", sample)){
        pt_size <- 0.1
    }
    if (grepl("HD", sample)) {
        p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = pt_size, override_plot_dims = TRUE, scale = "shared", ncol = length(features), shape = "tile")
        hd_sample_plots[[sample]] <- p & scale_fill_viridis_c(limits = c(0,1), option = "H")
    } else {
        p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = pt_size, override_plot_dims = TRUE, scale = "shared", ncol = length(features))
        sample_plots[[sample]] <- p & scale_fill_viridis_c(limits = c(0,1), option = "H")
    }
}

# Merge non-HD sample plots into a final plot
final_plot <- patchwork::wrap_plots(sample_plots, ncol = 1)
ggsave(paste0(wd, "map/", "sampleID_map.jpg"), final_plot, height = 8*5, width = 9*5, dpi = 300, limitsize = FALSE)

# Merge HD sample plots into a separate final plot
if (length(hd_sample_plots) > 0) {
    hd_final_plot <- patchwork::wrap_plots(hd_sample_plots, ncol = 1)
    ggsave(paste0(wd, "map/", "sampleID_mapHD.jpg"), hd_final_plot, height = 4*5, width = 9*5, dpi = 300, limitsize = FALSE)
}



## deconvolution of malignancy status



In [ ]:
samples <- unique(int.all$sample[!grepl("HD", int.all$sample)])
for (sample in samples){
    vis <- readRDS(paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/", sample, "_spatial.rds"))
    DefaultAssay(int.all) <- "RNA"

    ## Map w/ semla NNLS
    ident <- "malignancy" # ident in the reference to use for mapping
    celltype <- "malignancy_map" # name of the mapping to assign to the visium object
    DefaultAssay(vis) <- "Spatial"
    vis <- vis |> 
    NormalizeData() |>
    FindVariableFeatures(nfeatures = nrow(vis)) |>
    ScaleData()
    vis <- RunNNLS(vis, singlecell_object = int.all, spatial_assay = DefaultAssay(vis), singlecell_assay = DefaultAssay(int.all), groups = ident, slot = "data", assay_name = celltype)
    spls[[sample]]@assays[[celltype]] <- vis@assays[[celltype]]
    DefaultAssay(spls[[sample]]) <- celltype
}

# Plot each sample separately
sample_plots <- list()
hd_sample_plots <- list()

for (sample in samples) {
    vis <- spls[[sample]]
    features = rownames(vis@assays$malignancy_map)
    if (length(vis$sample_id) > 5000) { # 11*11 Array Visium
        pt_size <- 1
    } else if (length(vis$sample_id) < 5000) { # 6.5*6.5 Array Visium
        pt_size <- 1.6
    } else if (grepl("HD", sample)){
        pt_size <- 0.1
    }
    if (grepl("HD", sample)) {
        p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = pt_size, override_plot_dims = TRUE, scale = "shared", ncol = length(features), shape = "tile")
        hd_sample_plots[[sample]] <- p & scale_fill_viridis_c(limits = c(0,1), option = "H")
    } else {
        p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = pt_size, override_plot_dims = TRUE, scale = "shared", ncol = length(features))
        sample_plots[[sample]] <- p & scale_fill_viridis_c(limits = c(0,1), option = "H")
    }
}

# Merge non-HD sample plots into a final plot
final_plot <- patchwork::wrap_plots(sample_plots, ncol = 1)
ggsave(paste0(wd, "map/", "malignancy_map.jpg"), final_plot, height = 8*5, width = 9*5, dpi = 300, limitsize = FALSE)

# Merge HD sample plots into a separate final plot
if (length(hd_sample_plots) > 0) {
    hd_final_plot <- patchwork::wrap_plots(hd_sample_plots, ncol = 1)
    ggsave(paste0(wd, "map/", "malignancy_mapHD.jpg"), hd_final_plot, height = 4*5, width = 9*5, dpi = 300, limitsize = FALSE)
}



### save progress


In [ ]:
for (sample in samples){
    output_path <- paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/", sample, "_3decon.rds")
    saveRDS(spls[[sample]], file = output_path)
}



### load progress


In [ ]:
int.all <- readRDS(paste0(rootObj, "SIMPlex_breast_allSamples.rds"))
samples <- unique(int.all$sample[!grepl("HD", int.all$sample)])
spls <- list()
for (sample in samples) {
    output_path <- paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/", sample, "_3decon.rds")
    if (file.exists(output_path)) {
        spls[[sample]] <- readRDS(output_path)
    }
}



## compare proportions with histopathology 

### load histopath into corresponding samples


In [ ]:
# not all samples have histopathology data. include only the samples that do
samples_pathAnno <- samples[!samples %in% "patient2_55um"]
for (sample in samples_pathAnno) {
    types <- c("epithelial", "immune", "stroma", "additional")
    annot <- list()
    for (type in types) {
        csv_path <- file.path(HISTO_DIR, sample, paste0(type, ".csv"))
        if (file.exists(csv_path)) {
            annot[[type]] <- read.csv(csv_path)
        } else {
            annot[[type]] <- NULL
        }

        rownames(annot[[type]]) <- annot[[type]]$Barcode
        annot[[type]] <- annot[[type]][rownames(annot[[type]]) %in% rownames(spls[[sample]]@meta.data), ]
        annot[[type]][annot[[type]] == ""] <- NA  # Replace empty cells with NA
        spls[[sample]] <- AddMetaData(spls[[sample]], metadata = annot[[type]], col.name = colnames(annot[[type]])[2])
    }
}



#### plot annotations as they are


In [ ]:
for (sample in samples_pathAnno){
    vis <- spls[[sample]]
    if (length(vis$sample_id) > 5000) { # 11*11 Array Visium
        pt_size <- 1
    } else if (length(vis$sample_id) < 5000) { # 6.5*6.5 Array Visium
        pt_size <- 1.6
    } else if (grepl("HD", sample)){
        pt_size <- 0.1
    }
    p_list <- list()
    for (type in types) {
        p_list[[type]] <- MapLabels(vis, column_name = type, ncol = 1, label_by = "sample_id", override_plot_dims = TRUE, drop_na = TRUE, pt_size = pt_size)
    }
    p <- patchwork::wrap_plots(p_list, ncol = length(types)) & guides(fill = guide_legend(override.aes = list(size = 6), ncol = 1)) & theme(legend.position = "right")
    ggsave(paste0(wd, "map_pathAnnotation/", sample, "_annotation.jpg"), p, height = 5, width = length(types) * 5, dpi = 300, limitsize = FALSE)
}



#### simplified annotations


In [ ]:
for (sample in samples_pathAnno) {
    vis <- spls[[sample]]
    for (type in types) {
        simplified_column <- paste0(type, "_simplified")
        vis@meta.data[[simplified_column]] <- ifelse(!is.na(vis@meta.data[[type]]), type, NA)
    }
    spls[[sample]] <- vis
    if (length(vis$sample_id) > 5000) { # 11*11 Array Visium
        pt_size <- 1
    } else if (length(vis$sample_id) < 5000) { # 6.5*6.5 Array Visium
        pt_size <- 1.6
    } else if (grepl("HD", sample)){
        pt_size <- 0.1
    }
    p_list <- list()
    for (type in types) {
        p_list[[type]] <- MapLabels(vis, column_name = paste0(type, "_simplified"), ncol = 1, label_by = "sample_id", override_plot_dims = TRUE, drop_na = TRUE, pt_size = pt_size)
    }
    p <- patchwork::wrap_plots(p_list, ncol = length(types)) & guides(fill = guide_legend(override.aes = list(size = 6), ncol = 1)) & theme(legend.position = "right")
    ggsave(paste0(wd, "map_pathAnnotation/", sample, "_annotation_simplified.jpg"), p, height = 5, width = length(types) * 5, dpi = 300, limitsize = FALSE)
}



#### rework annotations


In [ ]:
for (sample in samples_pathAnno) {
    spls[[sample]]$stroma_simplified_filtered <- ifelse(
        !is.na(spls[[sample]]$stroma_simplified) & is.na(spls[[sample]]$epithelial),
        spls[[sample]]$stroma_simplified,
        NA
    )
}
for (sample in samples_pathAnno) {
    spls[[sample]]$immune_simplified_filtered <- ifelse(
        !is.na(spls[[sample]]$immune_simplified) & is.na(spls[[sample]]$stroma_simplified),
        spls[[sample]]$immune_simplified,
        NA
    )
}
for (sample in samples_pathAnno){
    vis <- spls[[sample]]
    if (length(vis$sample_id) > 5000) { # 11*11 Array Visium
        pt_size <- 1
    } else if (length(vis$sample_id) < 5000) { # 6.5*6.5 Array Visium
        pt_size <- 1.6
    } else if (grepl("HD", sample)){
        pt_size <- 0.1
    }
    p_list <- list()
    for (type in types) {
        if (type == "stroma") {
            p_list[[type]] <- MapLabels(vis, column_name = "stroma_simplified_filtered", ncol = 1, label_by = "sample_id", override_plot_dims = TRUE, drop_na = TRUE, pt_size = pt_size)
        } else {
            p_list[[type]] <- MapLabels(vis, column_name = paste0(type, "_simplified"), ncol = 1, label_by = "sample_id", override_plot_dims = TRUE, drop_na = TRUE, pt_size = pt_size)
        }
    }
    p <- patchwork::wrap_plots(p_list, ncol = length(types)) & guides(fill = guide_legend(override.aes = list(size = 6), ncol = 1)) & theme(legend.position = "right")
    ggsave(paste0(wd, "map_pathAnnotation/", sample, "_annotation_simplified_filtered.jpg"), p, height = 5, width = length(types) * 5, dpi = 300, limitsize = FALSE)
}
for (sample in samples_pathAnno) {
    spls[[sample]]$annot_reworked_simplified <- ifelse(
        !is.na(spls[[sample]]$epithelial_simplified), "Epithelial",
        ifelse(
            !is.na(spls[[sample]]$stroma_simplified_filtered), "Stroma",
            ifelse(
                !is.na(spls[[sample]]$immune_simplified_filtered), "Immune",
                NA
            )
        )
    )
}
for (sample in samples_pathAnno){
    vis <- spls[[sample]]
    if (length(vis$sample_id) > 5000) { # 11*11 Array Visium
        pt_size <- 1
    } else if (length(vis$sample_id) < 5000) { # 6.5*6.5 Array Visium
        pt_size <- 1.6
    } else if (grepl("HD", sample)){
        pt_size <- 0.1
    }
    p_list <- list()
        p_list[[type]] <- MapLabels(vis, column_name = "annot_reworked_simplified", ncol = 1, label_by = "sample_id", override_plot_dims = TRUE, drop_na = TRUE, pt_size = pt_size)
    p <- patchwork::wrap_plots(p_list, ncol = 1) & guides(fill = guide_legend(override.aes = list(size = 6), ncol = 1)) & theme(legend.position = "right")
    ggsave(paste0(wd, "map_pathAnnotation/", sample, "_reworked_simplified.jpg"), p, height = 5, width = 6, dpi = 300, limitsize = FALSE)
}
for (sample in samples_pathAnno) {
    spls[[sample]]$annot_reworked <- ifelse(
        !is.na(spls[[sample]]$epithelial), spls[[sample]]$epithelial,
        ifelse(
            !is.na(spls[[sample]]$stroma_simplified_filtered), spls[[sample]]$stroma,
            ifelse(
                !is.na(spls[[sample]]$immune_simplified_filtered), spls[[sample]]$immune,
                NA
            )
        )
    )
}
for (sample in samples_pathAnno){
    vis <- spls[[sample]]
    if (length(vis$sample_id) > 5000) { # 11*11 Array Visium
        pt_size <- 1
    } else if (length(vis$sample_id) < 5000) { # 6.5*6.5 Array Visium
        pt_size <- 1.6
    } else if (grepl("HD", sample)){
        pt_size <- 0.1
    }
    p_list <- list()
        p_list[[type]] <- MapLabels(vis, column_name = "annot_reworked", ncol = 1, label_by = "sample_id", override_plot_dims = TRUE, drop_na = TRUE, pt_size = pt_size)
    p <- patchwork::wrap_plots(p_list, ncol = 1) & guides(fill = guide_legend(override.aes = list(size = 6), ncol = 1)) & theme(legend.position = "right")
    ggsave(paste0(wd, "map_pathAnnotation/", sample, "_reworked.jpg"), p, height = 5, width = 6, dpi = 300, limitsize = FALSE)
}



#### save progress


In [ ]:
for (sample in samples_pathAnno){
    output_path <- paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/", sample, "_3decon.rds")
    saveRDS(spls[[sample]], file = output_path)
}



#### load progress


In [ ]:
int.all <- readRDS(paste0(rootObj, "SIMPlex_breast_allSamples.rds"))
samples <- unique(int.all$sample[!grepl("HD", int.all$sample)])
spls <- list()
for (sample in samples) {
    output_path <- paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/", sample, "_3decon.rds")
    if (file.exists(output_path)) {
        spls[[sample]] <- readRDS(output_path)
    }
}

In [ ]:
mergedVis_pathAnno <- MergeSTData(spls[!names(spls) %in% "patient2_55um"][[1]], spls[!names(spls) %in% "patient2_55um"][-1])
mergedVis_pathAnno <- subset(mergedVis_pathAnno, annot_reworked_simplified %in% na.omit(mergedVis_pathAnno$annot_reworked_simplified)) # remove NAs for missing path anno data



#### Stacked barplot of annotation, decon, and SN proportions 
get data.frame with all data 


In [ ]:
### --- A) BC Atlas scRNA --- ###
bc_atlas_counts <- table(bc_atlas$celltype_major)
bc_atlas_props <- data.frame(
    CellType = names(bc_atlas_counts),
    Proportion = as.numeric(bc_atlas_counts) / sum(bc_atlas_counts),
    Dataset = "BC_Atlas"
)

### --- B) Single nuclei --- ###
sn_count <- table(int.all$population)
sn_props <- data.frame(
    CellType = names(sn_count),
    Proportion = as.numeric(sn_count) / sum(sn_count),
    Dataset = "SingleNuclei"
)

### --- C) Spatial Deconvolution (patspec + bc_atlas) --- ###
patspec_mat <- mergedVis_pathAnno@assays$patspec_major_map@data
bc_atlas_mat  <- mergedVis_pathAnno@assays$garvan_major_map@data

patspec_counts <- rowSums(patspec_mat)
bc_atlas_counts_sp <- rowSums(bc_atlas_mat)

patspec_props <- data.frame(
    CellType = names(patspec_counts),
    Proportion = patspec_counts / sum(patspec_counts),
    Dataset = "Spatial_patspec"
)

bc_atlas_sp_props <- data.frame(
    CellType = names(bc_atlas_counts_sp),
    Proportion = bc_atlas_counts_sp / sum(bc_atlas_counts_sp),
    Dataset = "Spatial_BC_Atlas"
)

### --- D) Pathology annotation proportions --- ###
path_counts <- table(mergedVis_pathAnno$annot_reworked_simplified)

path_props <- data.frame(
    CellType = names(path_counts),
    Proportion = as.numeric(path_counts) / sum(path_counts),
    Dataset = "Pathology"
)

all_props <- bind_rows(
    bc_atlas_props,
    sn_props,
    patspec_props,
    bc_atlas_sp_props,
    path_props
)



###### Plot 


In [ ]:
# Recode datasets (unchanged)
all_props$Dataset <- dplyr::recode(
  all_props$Dataset,
  "BC_Atlas"   = "BC Atlas",
  "Pathology"      = "Histopathology",
  "SingleNuclei"   = "SIMPlex",
  "Spatial_BC_Atlas" = "Deconv. BC Atlas",
  "Spatial_patspec"= "Deconv. SIMPlex"
)

# Reorder datasets (new order) (unchanged)
all_props$Dataset <- factor(
  all_props$Dataset,
  levels = c(
    "Histopathology",
    "BC Atlas",
    "SIMPlex",
    "Deconv. BC Atlas",
    "Deconv. SIMPlex"
  )
)

# Reorder cell types (Epithelial on the bottom) (unchanged)
all_props$CellType <- factor(
  all_props$CellType,
  levels = c(
    "Epithelial",
    "Stroma",
    "CAFs",
    "Myeloid",
    "T-cells",
    "B-cells",
    "Plasmablasts",
    "Endothelial",
    "Adipocytes",
    "Immune",
    "PVL"
  )
)

# Colors stay the same (unchanged)
celltype_colors <- c(
  "Adipocytes"   = "#ED968C",
  "B-cells"      = "#2F8AC4",
  "CAFs"         = "#C49A50",
  "Endothelial"  = "#66A61E",
  "T-cells"      = "#1B9E77",
  "Immune"       = "#80B1D3",
  "Myeloid"      = "#386CB0",
  "Plasmablasts" = "#7FC97F",
  "PVL"          = "#BEAED4",
  "Stroma"       = "#CC79A7",
  "Epithelial"   = "#a90f11ff"
)

# --- Define x positions manually (ADJUSTED FOR EQUIDISTANT GROUP GAPS) ---
all_props$DatasetPos <- dplyr::recode(
  all_props$Dataset,
  "Histopathology"    = 1,
  "BC Atlas"          = 1.75, 
  "SIMPlex"           = 2.25, 
  "Deconv. BC Atlas"  = 3,  
  "Deconv. SIMPlex"   = 3.5
)
# -----------------------------------------------------------------------

# Plot
p <- ggplot(all_props, aes(x = DatasetPos, y = Proportion, fill = CellType)) +
  geom_bar(stat = "identity", width = 0.45, color = "black", linewidth = 0.23) +
  scale_x_continuous(
    # --- Update breaks with new positions ---
    breaks = c(1, 1.75, 2.25, 3, 3.5), 
    labels = c("Histopathology", "BC Atlas", "SIMPlex", "Deconv. BC Atlas", "Deconv. SIMPlex")
  ) +
  scale_fill_manual(values = celltype_colors) +
  ylab("Proportion") +
  xlab("") +
  theme_classic(base_size = 14) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 12)
  )

# Save
ggsave(
  filename = paste0(wd, "final_plots/data_prop_comparison.pdf"),
  plot = p,
  height = 5,
  width = 7,
  dpi = 300
)



#### heatmap of deconvolution per pathology annotation area 



In [ ]:
assays <- c("patspec_major_map","garvan_major_map","control_major_map")
meta <- mergedVis_pathAnno@meta.data
features <- c("Epithelial", "CAFs", "T-cells", "B-cells", "Plasmablasts", "Myeloid", "Endothelial", "PVL", "Adipocytes")  # Ensure consistent feature order
avg <- AverageExpression(mergedVis_pathAnno, features = features, group.by = "annot_reworked_simplified", slot = "data", assays = assays)
# Combine the matrices into a single long dataframe
prop_df <- lapply(names(avg), function(assay){
  mat <- avg[[assay]]
  df <- as.data.frame(as.matrix(mat))
  df$celltype <- rownames(df)
  df_long <- df %>%
    pivot_longer(cols = -celltype, names_to = "label", values_to = "prop") %>%
    mutate(assay = assay)
  df_long
}) %>% bind_rows()

# order cell types
cell_order <- c("Epithelial","CAFs","T-cells","B-cells","Plasmablasts","Myeloid","Endothelial","PVL","Adipocytes")
prop_df$celltype <- factor(prop_df$celltype, levels = cell_order)
# order annotations
prop_df$label <- factor(prop_df$label, levels = c("Epithelial", "Stroma", "Immune"))
assay_names <- c(
  "patspec_major_map" = "SIMPlex",
  "garvan_major_map"  = "BC atlas",
  "control_major_map" = "Technical Control"
)
prop_df$assay <- factor(prop_df$assay, levels = names(assay_names), labels = assay_names)
# Plot heatmap
p <- ggplot(prop_df, aes(x = label, y = celltype, fill = prop)) +
    geom_tile(color = "white") +
    geom_text(aes(label = sprintf("%.2f", prop)), size = 3) +
    facet_wrap(~ assay, nrow = 1) +
    scale_fill_gradient(low = "white", high = "steelblue") +
    scale_fill_gradientn(colors = colorRampPalette(c("white", "#ef9a38ff", "firebrick3"))(666)) +
    theme_minimal(base_size = 14) +
    theme(
        axis.title = element_blank(),
        axis.text.x = element_text(angle = 45, hjust = 1),
        strip.background = element_blank(),
        strip.text = element_text(face = "bold", size = 8),   # title size 
        panel.grid = element_blank()
    ) +
    labs(fill = "Proportion")

pdf(paste0(wd, "map_pathAnnotation/heatmap_histopath_decon.pdf"), width = 5.7, height = 4)
print(p)
dev.off()



## Comparison with automated Pathology - CTA (KI Xinsong collaboration)
This automated AI pathology pipeline generates pixel level annotation and we get and feed spot level proportions into our 
seurat objects

### load annotations


In [ ]:
samples_autoAnno <- unique(int.all$sample[!grepl("HD", int.all$sample) & !grepl("patient1", int.all$sample) & !grepl("patient2", int.all$sample)])
for (sample in samples_autoAnno) {
    annot <- read.csv(paste0(CTA_DIR, "/", sample, ".csv"))
    rownames(annot) <- annot$cellid
    annot <- annot[rownames(annot) %in% rownames(spls[[sample]]@meta.data), ]
    # Create an assay from the annotation columns
    valid_columns <- intersect(c("Stroma", "Tumor", "Normal_gland", "Immune", "Apoptosis", "total", "tumor_per", "immune_per", "stroma_per", "normal_gland_per"), colnames(annot))
    annot_assay <- CreateAssayObject(counts = t(as.matrix(annot[, valid_columns])))
    spls[[sample]][["autoAnno"]] <- annot_assay
}

In [ ]:
spls <- spls[!names(spls) %in% c("patient1_55um", "patient2_55um")]
mergedVis_autoAnno <- MergeSTData(spls[[1]], spls[-1])

In [ ]:
for (sample in samples_autoAnno){
    vis <- spls[[sample]]
    DefaultAssay(vis) <- "autoAnno"
    if (length(vis$sample_id) > 5000) { # 11*11 Array Visium
        pt_size <- 1
    } else if (length(vis$sample_id) < 5000) { # 6.5*6.5 Array Visium
        pt_size <- 1.6
    } else if (grepl("HD", sample)){
        pt_size <- 0.1
    }
    features <- paste0("autoanno_", rownames(vis@assays$autoAnno))
    vis <- LoadImages(vis)
    p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = pt_size, override_plot_dims = TRUE, scale = "shared", ncol = length(features), image_use = "raw")
    ggsave(paste0(wd, "autoAnnotation/", sample, "_autoAnno.jpg"), p, height = 5, width = length(features) * 5, dpi = 300, limitsize = FALSE)
}



#### correlate CTA annotation to decon 


In [ ]:
#### get correlation list 
assays <- c("patspec_major_map", "control_major_map", "garvan_major_map")
cors <- list()
for (assay in assays){
    DefaultAssay(mergedVis_autoAnno) <- assay
    data1 <- FetchData(mergedVis_autoAnno, rownames(mergedVis_autoAnno))
    DefaultAssay(mergedVis_autoAnno) <- "autoAnno"
    data2 <- FetchData(mergedVis_autoAnno, paste0("autoanno_", rownames(mergedVis_autoAnno@assays$autoAnno)))
    # Custom order for data1 and data2
    custom_order_data1 <- c("PVL", "Endothelial", "CAFs", "Epithelial", "Myeloid", "Plasmablasts", "T-cells", "B-cells")
    custom_order_data2 <-  paste0("autoanno_", c("Immune", "Normal-gland", "Tumor", "Stroma"))

    
    # Intersect cells of data1 and data2
    shared_cells <- intersect(rownames(data1), rownames(data2))
    data1 <- data1[shared_cells, , drop = FALSE]
    data2 <- data2[shared_cells, , drop = FALSE]ghu
    
    # Reorder columns based on custom order
    data1 <- data1[, custom_order_data1, drop = FALSE]
    data2 <- data2[, custom_order_data2, drop = FALSE]
    cors[[paste(assay, collapse = "")]] <- cor(data1, data2, method = "spearman")
}
for (name in names(cors)) {
    cor_matrix <- cors[[name]]
    cor_matrix_name <- name  # Use the name directly
    # Generate heatmap and save as PDF
    pdf(paste0(wd, "autoAnnotation/autoAnno_vs_", cor_matrix_name, ".pdf"), width = 10, height = 10)
    heatmap3(
        cor_matrix, 
        margins = c(20, 12), 
        symm = FALSE,
        Rowv = NA, 
        Colv = NA
    )
    dev.off()  # Close the PDF device
}



##### Boxplot of autoanno-decon correlation



In [ ]:
correspondences <- list(
    "B-cells" = "autoanno_Immune",
    "T-cells" = "autoanno_Immune",
    "Plasmablasts" = "autoanno_Immune",
    "Myeloid" = "autoanno_Immune",
    "Epithelial" = c("autoanno_Tumor", "autoanno_Normal-gland"),
    "CAFs" = "autoanno_Stroma",
    "Endothelial" = "autoanno_Stroma",
    "PVL" = "autoanno_Stroma"
)

correlation_values <- list()
for (assay in names(cors)) {
    cor_matrix <- cors[[assay]]
    assay_correlations <- sapply(names(correspondences), function(celltype) {
        modules <- correspondences[[celltype]]
        if (celltype %in% rownames(cor_matrix)) {
            valid_modules <- modules[modules %in% colnames(cor_matrix)]
            if (length(valid_modules) > 0) {
                return(mean(cor_matrix[celltype, valid_modules], na.rm = TRUE))
            }
        }
        return(NA)
    })
    correlation_values[[assay]] <- assay_correlations
}

total_correlation_scores <- sapply(correlation_values, function(assay_correlations) {
    mean(assay_correlations, na.rm = TRUE)
})

# Create a data frame for visualization
correlation_df <- do.call(rbind, lapply(names(correlation_values), function(assay) {
    data.frame(
        Assay = assay,
        CellType = names(correlation_values[[assay]]),
        Correlation = correlation_values[[assay]]
    )
}))

# Add mean correlation for each assay
mean_correlation_df <- data.frame(
    Assay = names(total_correlation_scores),
    CellType = "Mean",
    Correlation = total_correlation_scores
)

# Combine both data frames
mean_correlation_df$Assay <- recode(mean_correlation_df$Assay, 
                                    "control_major_map" = "Neg.\n Control", 
                                    "garvan_major_map" = "Ref.\n Atlas", 
                                    "patspec_major_map" = "SIMPlex")
correlation_df$Assay <- recode(correlation_df$Assay, 
                               "control_major_map" = "Neg.\n Control", 
                               "garvan_major_map" = "Ref.\n Atlas", 
                               "patspec_major_map" = "SIMPlex")
correlation_df <- rbind(correlation_df, mean_correlation_df)

# Plot
plot <- ggplot(correlation_df, aes(x = Assay, y = Correlation)) +
    geom_boxplot(data = subset(correlation_df, CellType != "Mean"),
                 aes(fill = Assay), alpha = 0.25, color = "black", outlier.shape = NA) +  # Boxplot for distribution
    geom_jitter(data = subset(correlation_df, CellType != "Mean"),
                aes(color = CellType), width = 0.2, height = 0, size = 2) +  # Cell types
    ggrepel::geom_text_repel(data = subset(correlation_df, CellType != "Mean" & 
                                           (Correlation > quantile(Correlation, 0) | 
                                            Correlation < quantile(Correlation, 1))),
                             aes(label = CellType, color = CellType), 
                             size = 2, 
                             box.padding = 0.2, 
                             point.padding = 0.1, 
                             max.overlaps = 5) +  # Annotate outlier cell types with repelling text
    scale_color_manual(values = colsCelltype[-1]) +  # Use custom colors, ignoring the first color
    scale_fill_manual(values = colsAssays) +  # Use custom colors for Assay
    labs(title = "Deconv. correlation to CTA annotation",
         x = "Assay", y = "Spearman Correlation Coeff.") +
    theme_minimal() +
    theme(legend.title = element_blank(),
          panel.grid = element_blank(),
          axis.line = element_line(color = "black"),  # Add axis lines
          axis.ticks = element_line(color = "black"))  # Add axis ticks

ggsave(
    filename = paste0(wd, "autoAnnotation/boxplot_autoAnno_vs_decon.pdf"),
    plot = plot,
    width = 4,
    height = 5
)



## Visium clusterdness versus simplex 

#### load progress


In [ ]:
int.all <- readRDS(paste0(rootObj, "SIMPlex_breast_allSamples.rds"))
samples <- unique(int.all$sample[!grepl("HD", int.all$sample)])
spls <- list()
for (sample in samples) {
    output_path <- paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/", sample, "_3decon.rds")
    if (file.exists(output_path)) {
        spls[[sample]] <- readRDS(output_path)
    }
}



### get merged visium and integrate


In [ ]:
mergedVis <- MergeSTData(spls[[1]], spls[-1])

In [ ]:
set.seed(1)
DefaultAssay(mergedVis) <- "Spatial"
mergedVis <- mergedVis %>%
  NormalizeData() %>%
  FindVariableFeatures() %>%
  ScaleData() %>%
  RunPCA() %>%
  RunUMAP(dims = 1:30)
# plot pre integration
ggsave(
    filename = paste0(wd, "integration_visium/pca_pre_integration.pdf"),
    plot = DimPlot(mergedVis, reduction = "pca", group.by = "sample_id"),
    width = 10,
    height = 10
)
ggsave(
    filename = paste0(wd, "integration_visium/umap_pre_integration.pdf"),
    plot = DimPlot(mergedVis, reduction = "umap", group.by = c("sample_id"), ncol = 1),
    width = 10,
    height = 10
)
# integrate
int.vis <- mergedVis %>% 
    RunHarmony("sample_id", plot_convergence = TRUE, theta = 0) # min thetha since we expect a lot of biological variation
int.vis <- RunUMAP(int.vis, dims = 1:30, reduction = "harmony")

# plot post integration
ggsave(
    filename = paste0(wd, "integration_visium/pca_post_integration.pdf"),
    plot = DimPlot(int.vis, reduction = "harmony", group.by = "sample_id"),
    width = 10,
    height = 10
)
ggsave(
    filename = paste0(wd, "integration_visium/umap_post_integration.pdf"),
    plot = DimPlot(int.vis, reduction = "umap", group.by = c("sample_id"), ncol = 1),
    width = 10,
    height = 10
)



### get clusters at multiple resolutions for both SIMPlex and Visium data 


In [ ]:
set.seed(1)
obj <- int.vis ### 
DefaultAssay(obj) <- "RNA" ###
plot <- FALSE ### 
obj <- FindNeighbors(obj, dims = 1:30)
for (res in c(0.1, 0.25, .5, 0.8, 1, 1.25)){
  obj <- FindClusters(obj, resolution = res, algorithm = 1, verbose = FALSE)
}

d1 <- DimPlot(obj, reduction = "umap", group.by = paste0(DefaultAssay(obj), "_snn_res.0.1"), label = TRUE) + 
    DimPlot(obj, reduction = "umap", group.by = paste0(DefaultAssay(obj), "_snn_res.0.25"), label = TRUE) + 
    DimPlot(obj, reduction = "umap", group.by = paste0(DefaultAssay(obj), "_snn_res.0.5"), label = TRUE)

d2 <- DimPlot(obj, reduction = "umap", group.by = paste0(DefaultAssay(obj), "_snn_res.0.8"), label = TRUE) + 
    DimPlot(obj, reduction = "umap", group.by = paste0(DefaultAssay(obj), "_snn_res.1"), label = TRUE) + 
    DimPlot(obj, reduction = "umap", group.by = paste0(DefaultAssay(obj), "_snn_res.1.25"), label = TRUE)

if (plot){
    pdf(paste0(wd, "integration/.pdf"), width = 20, height = 10)
    d1 / d2
    dev.off()
}
int.vis <- obj ### 



### separate based UMAP identity regions: malignant and non malignant

##### label transfer malignancy from SIMPlex data


In [ ]:
cluster_res <- "Spatial_snn_res.0.5"
ref <- int.all
DefaultAssay(ref) <- "RNA"
obj <- int.vis
obj <- JoinLayers(obj, assay = "Spatial")
DefaultAssay(obj) <- "Spatial"

obj <- obj |>
  NormalizeData() |>
  FindVariableFeatures() |>
  ScaleData()

anchors <- FindTransferAnchors(reference = ref, query = obj)
predictions <- TransferData(
  anchorset = anchors,
  refdata = ref$,
  verbose = FALSE
)
obj <- AddMetaData(object = obj, metadata = predictions)

annotation_plot <- DimPlot(obj, group.by = "predicted.id", label = TRUE)
cluster_plot <- DimPlot(obj, group.by = cluster_res, label = TRUE)
ggsave(
    filename = paste0(wd, "integration_visium/malignancy_labeltransfer.pdf"),
    plot = cluster_plot + annotation_plot,
    width = 20,
    height = 10
)

obj$malignancy <- obj$predicted.id
int.vis <- obj

In [ ]:
dataname <- "sample_id"
int_data <- table(int.vis$sample_id)
cols <- colsSample

data <- data.frame(
    Category = dataname,
    Group = names(int_data),
    Nuclei = as.numeric(int_data),
    Proportion = as.numeric(int_data) / sum(as.numeric(int_data))
)

# Create the bar plot
p1 <- ggplot(data) + 
    aes(x = Category, y = Proportion, fill = Group) +
    geom_bar(stat = "identity", position = "stack") +
    ylab("Proportions") +
    geom_text(aes(label = Nuclei), 
              position = position_stack(vjust = 0.5), size = 3) +
    ggtitle("") +
    theme_bw() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) +  
    annotate("text", x = 1, y = 1.05, label = paste0("N=", length(unique(int.vis$sample_id)), "\nNspots=", sum(as.numeric(int_data))), size = 3) + 
    scale_fill_manual(values = cols)

#### UMAP with same colors 
# Combine the bar plot and UMAP plot using patchwork
umap_plot <- DimPlot(int.vis, reduction = "umap", group.by = dataname, label = FALSE, alpha = 0.2, pt.size = 0.001, cols = cols)
combined_plot <- p1 + umap_plot + plot_layout(widths = c(0.2, 0.8))

ggsave(
        filename = paste0(wd, "integration_visium/", dataname, "_proportions_umap_visium.pdf"),
        plot = combined_plot,
        width = 9,
        height = 6
)



### Calculate sillouette score
Since this is rather computationally intensive we subsample the data to 500 cells per "ident"


In [ ]:
# Calculate silhouette scores for all resolutions and store in a list
library(cluster)

calculate_silhouette <- function(obj, resolutions, ident, reduction = "harmony", dims = 1:30, metadata_col = "malignancy") {
    sil_list <- list()
    Idents(obj) <- ident
    sub <- subset(obj, downsample = 500)
    for (res in resolutions) {
        dist.matrix <- dist(x = Embeddings(object = sub[[reduction]])[, dims])
        clusters <- sub[[paste0(DefaultAssay(obj), "_snn_res.", res)]][[1]]
        sil <- silhouette(x = as.numeric(as.factor(clusters)), dist = dist.matrix)
        sil_df <- data.frame(
            silhouette = sil[, 3],
            cluster = clusters,
            metadata = sub[[metadata_col]][[1]]
        )
        sil_list[[paste0("res_", res)]] <- sil_df
    }
    return(sil_list)
}

resolutions <- c(0.1, 0.25, 0.5, 0.8, 1, 1.25)
silhouette_scores <- list(
    SIMPlex = calculate_silhouette(int.all, resolutions, "sample", metadata_col = "malignancy"),
    Visium_55um = calculate_silhouette(int.vis, resolutions, "sample_id", metadata_col = "malignancy")
)

# Save silhouette scores to an RDS file
saveRDS(silhouette_scores, file = paste0(rootObj, "silhouette_scores.rds"))



### boxplot per resolution


In [ ]:
prepare_silhouette_df <- function(assay_scores, assay_name) {
    assay_scores %>%
        enframe(name = "resolution", value = "silhouette") %>%
        mutate(
            resolution = gsub("res_", "", resolution),
            resolution = as.numeric(resolution)
        ) %>%
        unnest(cols = c(silhouette)) %>%
        mutate(assay = assay_name)
}

# Prepare data for both assays
df_simpl <- prepare_silhouette_df(silhouette_scores$SIMPlex, "SIMPlex")
df_visium <- prepare_silhouette_df(silhouette_scores$Visium_55um, "Visium_55um")

# Combine into one data frame
df_all <- bind_rows(df_simpl, df_visium)

# Plot for each assay
p <- ggplot(df_all, aes(x = factor(resolution), y = silhouette, fill = assay)) +
        geom_boxplot(outlier.shape = NA, alpha = 0.7) +
        facet_wrap(~ assay) +
        labs(
                title = "Silhouette Scores Across Resolutions",
                x = "Resolution",
                y = "Silhouette Score"
        ) +
        theme_minimal() +
        theme(legend.position = "none")

ggsave(
        filename = paste0(wd, "integration_visium/silhouette_res_boxplot.pdf"),
        plot = p,
        width = 10,
        height = 5
)



### boxplot per malignancy


In [ ]:
# Prepare data for both assays pooling all resolutions
prepare_silhouette_metadata_df <- function(assay_scores, assay_name) {
    assay_scores %>%
        enframe(name = "resolution", value = "silhouette") %>%
        unnest(cols = c(silhouette)) %>%
        mutate(assay = assay_name)
}

# Prepare data for both assays
df_simpl <- prepare_silhouette_metadata_df(silhouette_scores$SIMPlex, "SIMPlex")
df_visium <- prepare_silhouette_metadata_df(silhouette_scores$Visium_55um, "Visium_55um")

# Combine into one data frame
df_all <- bind_rows(df_simpl, df_visium)

# Plot pooling all resolutions
p <- ggplot(df_all, aes(x = metadata, y = silhouette, fill = assay)) +
    geom_violin(alpha = 0.7, scale = "width", trim = FALSE, position = position_dodge(width = 0.85)) +
    stat_summary(fun = median, geom = "crossbar", width = 0.5, color = "black", position = position_dodge(width = 0.85)) +
    scale_fill_manual(values = c("#20B2AA", "#6A5ACD")) +
    labs(
        title = "Silhouette Scores Across Path. Status (All Clustering Resolutions)",
        x = "Path. Status",
        y = "Silhouette Score"
    ) +
    theme_minimal() +
    theme(
        axis.text.x = element_text(angle = 45, hjust = 1, margin = margin(t = 10))
    )

ggsave(
        filename = paste0(wd, "integration_visium/silhouette_malignancy_violin_all_resolutions.pdf"),
        plot = p,
        width = 7,
        height = 4
)



## Visium HD progressive mapping with diff resolutions 
running on both prostate and breast data 

##### load visium HD data and save object


In [ ]:
library(stringr)
library(arrow)
samples <- c("pt10", "pt20")
spls <- list()
for (sample in samples){
st.dir <- paste0(SPACERANGER, "/prostate_cancer/HD/", sample, "_HD/binned_outputs")
res <- paste(0, c(2, 8, 16), "um", sep = "")
all.res <- paste(res, collapse = "|")
res.dir <- list.dirs(st.dir, recursive = FALSE) |> 
  stringr::str_subset(pattern = all.res)
infoTable <- data.frame(
  samples = list.files(res.dir,
                       full.names = TRUE, recursive = TRUE,
                       pattern = paste0("^filtered_feature.+.h5$")
  ),
  spotfiles = list.files(res.dir,
                         full.names = TRUE, recursive = TRUE,
                         pattern = "parquet$|positions.csv$"
  ),
  imgs = list.files(res.dir,
                    recursive = TRUE,
                    full.names = TRUE, pattern = "hires"
  ) |>
    str_subset(all.res),
  json = list.files(st.dir,
                    recursive = TRUE,
                    full.names = TRUE, pattern = "^scalefactors"
  ) |>
    str_subset(all.res),
  resolution = res,
  sample_id = sample
)
se.hd <- ReadVisiumData(infoTable)
dir.create(paste0(SPATIAL_RDS, "/prostate_cancer/", sample, "_visHD/"), recursive = TRUE, showWarnings = FALSE)
saveRDS(se.hd, paste0(SPATIAL_RDS, "/prostate_cancer/", sample, "_visHD/", sample, "_visHD.rds"))
}




### plot diff references to HD data for patient 5 



In [ ]:
patient <- "patient5"
ress <- c("s_002um", "s_008um", "s_016um", "s_048um")
control <- subset(int.all, subset = patient_ID != !!patient)
simp <- subset(int.all, subset = patient_ID == patient)
bc_atlas$major_celltypes <- bc_atlas$celltype_major
refs <- list(bc_atlas = bc_atlas, SIMPlex = simp, control = control)

vis <- readRDS(paste0(SPATIAL_RDS, "/breast_cancer/HD/patient5_HD/patient5_HD_spatial.rds"))
subset48 <- SubsetSTData(vis, spots = grep("s_048um", colnames(vis), value = TRUE)) 
    DefaultAssay(subset48) <- "Spatial"
    subset48 <- subset48 |>
    NormalizeData() |>
    FindVariableFeatures(nfeatures = 2000) |>
    ScaleData()
var.features <- VariableFeatures(subset48)

for (res in ress) {
    barcodes_in_range <- grep(paste0("^", res), colnames(vis), value = TRUE)
    if (res == "s_002um") {
        barcodes_in_range <- vis@tools$Staffli@meta_data %>%
            filter(pxl_col_in_fullres >= max(pxl_col_in_fullres * 0.4), # left
                pxl_col_in_fullres <= max(pxl_col_in_fullres * 0.6), # right 
                pxl_row_in_fullres >= max(pxl_row_in_fullres * 0.45), # top
                pxl_row_in_fullres <= max(pxl_row_in_fullres * 0.65)) %>% # bottom
            pull(barcode) %>%
            grep(paste0("^", res), ., value = TRUE)
    }
    subRes <- SubsetSTData(vis, spots = barcodes_in_range)
    ## quick check of the frame
    ggsave(paste0(FIGS_ROOT, "/breast_cancer/analysis_majorLevel/HD_decon_comparison/patient5_area.jpg"), MapFeatures(subRes, features = "nFeature_Spatial", override_plot_dims = TRUE, pt_size = 0.5, shape = "tile"))
    ## Map w/ semla NNLS
    ident <- "major_celltypes" # ident in the reference to use for mapping 
    celltype <- paste0("major_map_re", res) # name of the mapping to assign to the visium object
    DefaultAssay(subRes) <- "Spatial"
        subRes <- subRes |>
        NormalizeData() |>
        ScaleData()
    VariableFeatures(subRes) <- var.features # we use bin 48's var features due to sparsity of HD 

    for (refName in c("bc_atlas", "SIMPlex", "control")){
        ref <- refs[[refName]]
        DefaultAssay(subRes) <- "Spatial"
        subRes <- RunNNLS(subRes, singlecell_object = ref, spatial_assay = DefaultAssay(subRes), singlecell_assay = DefaultAssay(ref), groups = ident, slot = "data", assay_name = celltype)
        DefaultAssay(subRes) <- celltype

        # Plot each resolution separately
        features <- unique(gsub("_", "-", ref$major_celltypes))
        features <- features[order(features)]  # Ensure consistent order
        p <- MapFeatures(subRes, features = features, colors = viridis::turbo(11), label_by = "sample_id", override_plot_dims = TRUE, scale = "shared", ncol = length(features), shape = "tile", drop_na = TRUE)
        ggsave(paste0(wd, "HD_decon_comparison/", patient,"_", refName ,"_", res, "_major_mapHD.jpg"), p, height = 10, width = 50, dpi = 300, limitsize = FALSE)
    }
}



### plot maximum celltype assignment per datapoint @16um


In [ ]:
res <- "s_016um"
crop <- TRUE
barcodes_in_range <- grep(paste0("^", res), colnames(vis), value = TRUE)
if (crop == TRUE) {
    barcodes_in_range <- vis@tools$Staffli@meta_data %>%
        filter(pxl_col_in_fullres >= max(pxl_col_in_fullres * 0.4), # left
            pxl_col_in_fullres <= max(pxl_col_in_fullres * 0.6), # right 
            pxl_row_in_fullres >= max(pxl_row_in_fullres * 0.45), # top
            pxl_row_in_fullres <= max(pxl_row_in_fullres * 0.65)) %>% # bottom
        pull(barcode) %>%
        grep(paste0("^", res), ., value = TRUE)
}
subRes <- SubsetSTData(vis, spots = barcodes_in_range)

## Map w/ semla NNLS
ident <- "major_celltypes" # ident in the reference to use for mapping 
celltype <- paste0("major_map_re", res) # name of the mapping to assign to the visium object
DefaultAssay(subRes) <- "Spatial"
    subRes <- subRes |>
    NormalizeData() |>
    ScaleData()
VariableFeatures(subRes) <- var.features # we use bin 48's var features due to sparsity of HD 

for (refName in c("bc_atlas", "SIMPlex", "control")){
    ref <- refs[[refName]]
    DefaultAssay(subRes) <- "Spatial"
    subRes <- RunNNLS(subRes, singlecell_object = ref, spatial_assay = DefaultAssay(subRes), singlecell_assay = DefaultAssay(ref), groups = ident, slot = "data", assay_name = celltype)
    

    # Plot each resolution separately
    features <- unique(gsub("_", "-", ref$major_celltypes))
    features <- features[order(features)]  # Ensure consistent order

        # get max value 
        mtx <- GetAssayData(subRes, assay = "major_map_res_016um", slot = "data")
        max_features <- apply(mtx, 2, function(x) {
        if (all(is.na(x))) {
            return(NA)  # No niche passes threshold
        } else {
            return(rownames(mtx)[which.max(x)])
        }
        })
        subRes$top_celltype <- unlist(max_features)

    cols <- if ("Adipocytes" %in% unique(subRes$top_celltype)) colsCelltype else colsCelltype[-1]
    p <- MapLabels(subRes, column_name = "top_celltype", colors = cols, override_plot_dims = TRUE, shape = "tile", drop_na = TRUE)
    if (crop) {
        ggsave(paste0(wd, "HD_decon_comparison/", patient,"_", refName , "_discretized_mapHD_zoomed.jpg"), p, height = 10, width = 10, dpi = 300, limitsize = FALSE)
    } else {
        ggsave(paste0(wd, "HD_decon_comparison/", patient,"_", refName , "_discretized_mapHD.jpg"), p, height = 10, width = 10, dpi = 300, limitsize = FALSE)
    }
}




# Calculate nuclei per spot metrics 

#### load progress


In [ ]:
int.all <- readRDS(paste0(rootObj, "SIMPlex_breast_allSamples.rds"))
samples <- unique(int.all$sample[!grepl("HD", int.all$sample)])
spls <- list()
for (sample in samples) {
    output_path <- paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/", sample, "_3decon.rds")
    if (file.exists(output_path)) {
        spls[[sample]] <- readRDS(output_path)
    }
}

In [ ]:
# Calculate the number of nuclei per spot for each patient
nuclei_per_spot <- data.frame()

for (sample in names(spls)) {
    vis <- spls[[sample]]
    patient <- unique(int.all$patient_ID[int.all$sample == sample])
    if (length(patient) == 1) {
        nuclei_count <- table(int.all$sample[int.all$patient_ID == patient])
        spot_count <- ncol(vis)
        nuclei_per_spot <- rbind(
            nuclei_per_spot,
            data.frame(
                Patient = patient,
                Sample = sample,
                Nuclei = nuclei_count[sample],
                Spots = spot_count,
                NucleiPerSpot = nuclei_count[sample] / spot_count
            )
        )
    }
}

# Save the results as a CSV
output_path <- paste0(wd, "nuclei_per_spot.csv")
write.csv(nuclei_per_spot, file = output_path, row.names = FALSE)



# regenerate some plots for the publication 

### name changes on the integrated object


In [ ]:
int.all$major_celltypes <- int.all$seuratAnnotation_major # renaming for interpretability
int.all$epithelial_status <- ifelse(int.all$malignancy == "Benign", "Non-epithelial", int.all$malignancy)
colsEpithelialStatus <- c(colsMalignancy[2], colsMalignancy[1], colsMalignancy[-c(1, 2)])



### major celltype marker genes 


In [ ]:
Idents(int.all) <- "major_celltypes"
DefaultAssay(int.all) <- "Spatial"

markers <- FindAllMarkers(int.all, only.pos = TRUE, min.pct = 0.25, logfc.threshold = 0.25)
top_markers <- markers %>%
  group_by(cluster) %>%
  top_n(n = 4, wt = avg_log2FC)

temp <- subset(int.all, major_celltypes %in% na.omit(Idents(int.all)))
p <- DotPlot(temp, features = unique(top_markers$gene)) + # the subset is to fix a bug related to NAs 
        coord_flip() +
        scale_color_gradientn(colors = c("blue", "white", "darkred")) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1))
ggsave(paste0(wd, "final_plots/major_markers_dotplot.pdf"), p, width = 6, height = 8)

In [ ]:
int.all$epithelial_status <- ifelse(int.all$malignancy == "Benign", "Non-epithelial", int.all$malignancy)
colsEpithelialStatus <- c(colsMalignancy[2], colsMalignancy[1], colsMalignancy[-c(1, 2)])



### regenerated UMAPs with different labels with stacked bar plots 


In [ ]:
dataname <- "epithelial_status"
int_data <- table(int.all$epithelial_status)
cols <- colsEpithelialStatus
label <- TRUE

data <- data.frame(
    Category = dataname,
    Group = names(int_data),
    Nuclei = as.numeric(int_data),
    Proportion = as.numeric(int_data) / sum(as.numeric(int_data))
)

# Create the bar plot
p1 <- ggplot(data) + 
    aes(x = Category, y = Proportion, fill = Group) +
    geom_bar(stat = "identity", position = "stack") +
    ylab("Proportions") +
    geom_text(aes(label = Nuclei), 
              position = position_stack(vjust = 0.5), size = 3) +
    ggtitle("") +
    theme_bw() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) +  
    annotate("text", x = 1, y = 1.05, label = paste0("N=", length(unique(int.all$sample)), "\nNnuclei=", sum(as.numeric(int_data))), size = 3) + 
    scale_fill_manual(values = cols)

#### UMAP with same colors 
# Combine the bar plot and UMAP plot using patchwork
umap_plot <- DimPlot(int.all, reduction = "umap", group.by = dataname, label = label, alpha = 0.2, pt.size = 0.001, cols = cols) +
    theme(legend.position = "none")
combined_plot <- p1 + umap_plot + plot_layout(widths = c(0.2, 0.8))

ggsave(
        filename = paste0(wd, "final_plots/", dataname, "_proportions_umap_V2.pdf"),
        plot = combined_plot,
        width = 8,
        height = 6
)



## Major decon proportions - collapsed bar plot 

#### load spatial 


In [ ]:
samples <- unique(int.all$sample[!grepl("HD", int.all$sample)])
spls <- list()
for (sample in samples) {
    if (!grepl("HD", sample)) {
        output_path <- paste0(paste0(SPATIAL_RDS, "/breast_cancer/55um/", sample, "/", sample, "_3decon.rds")
        if (file.exists(output_path)) {
            spls[[sample]] <- readRDS(output_path)
        }
    }
}
mergedVis <- MergeSTData(spls[[1]], spls[-1])

In [ ]:
# Extract matrices
patspec_mat <- mergedVis@assays$patspec_major_map@data
bc_atlas_mat  <- mergedVis@assays$garvan_major_map@data

# Sum across cells for each cell type 
patspec_counts <- rowSums(patspec_mat)
bc_atlas_counts  <- rowSums(bc_atlas_mat)

# Combine into data.frame
data <- data.frame(
    CellType = c(names(patspec_counts), names(bc_atlas_counts)),
    Count = c(as.numeric(patspec_counts), as.numeric(bc_atlas_counts)),
    Dataset = rep(c("SIMPlex", "BC Atlas"), 
                                times = c(length(patspec_counts), length(bc_atlas_counts)))
)

# Compute proportions per dataset
data <- data %>%
    group_by(Dataset) %>%
    mutate(Proportion = Count / sum(Count))

# Consistent CellType order across plots
celltype_order <- c("Adipocytes", "B-cells", "CAFs", "Endothelial", 
                    "Epithelial", "Myeloid", "Plasmablasts", "PVL", "T-cells")
data$CellType <- factor(data$CellType, levels = celltype_order)

# ==== Plot 1: Stacked bar per dataset ====

# Define your color palette
cols <- colsCelltype  # your custom palette

p1 <- ggplot(data) + 
    aes(x = Dataset, y = Proportion, fill = CellType) +
    geom_bar(stat = "identity", position = "stack") +
    geom_text(aes(label = paste0(round(Proportion * 100, 1), "%")), 
                        position = position_stack(vjust = 0.5), size = 3) +
    ylab("Deconv. Proportions") +
    ggtitle("Deconvolution Proportions") +
    theme_bw() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) + 
    scale_fill_manual(values = cols)

ggsave(paste0(wd, "final_plots/deconvolution_proportions_soft_deconv_stacked.pdf"), p1, width = 4, height = 7)

# ==== Plot 2: Side-by-side bars per cell type ====

cols2 <- rev(paletteer_c("grDevices::Temps", 2))

p2 <- ggplot(data) + 
    aes(x = CellType, y = Proportion, fill = Dataset) +
    geom_bar(stat = "identity", position = "dodge") +
    geom_text(aes(label = round(Count, 1)), 
                        position = position_dodge(width = 0.9), size = 3, vjust = -0.5) +
    ylab("Deconv. Proportions") +
    ggtitle("Deconvolution Comparison by Cell Type") +
    theme_bw() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) + 
    scale_fill_manual(values = cols2)

ggsave(paste0(wd, "final_plots/deconvolution_proportions_soft_deconv_dodge.pdf"), p2, width = 10, height = 5)




### get QC metrics



In [ ]:
int.all <- readRDS(paste0(SN_RDS, "/breast_cancer/cross_patient/SIMPlex_breast_allSamples.rds"))

In [ ]:
qc_medians <- int.all@meta.data %>%
    group_by(sample) %>%
    summarise(
        nFeature_RNA = median(nFeature_RNA, na.rm = TRUE),
        nCount_RNA = median(nCount_RNA, na.rm = TRUE)
    ) %>%
    tidyr::pivot_longer(
        cols = c(nFeature_RNA, nCount_RNA),
        names_to = "feature",
        values_to = "median_value"
    )

library(patchwork)
plots <- lapply(unique(qc_medians$feature), function(feat) {
    medians <- qc_medians %>% filter(feature == feat)
    # Get max value per sample for y position
    max_vals <- int.all@meta.data %>%
        group_by(sample) %>%
        summarise(max_y = max(.data[[feat]], na.rm = TRUE))
    medians <- left_join(medians, max_vals, by = "sample")
    # Calculate x positions: right of each violin
    sample_levels <- levels(factor(int.all@meta.data$sample))
    n_samples <- length(sample_levels)
    x_positions <- seq_along(sample_levels)  # shift right
    medians$x_pos <- x_positions[match(medians$sample, sample_levels)]
    # Set y axis limit to 1.2x the max value
    y_max <- max(medians$max_y, na.rm = TRUE) * 1.2
    p <- VlnPlot(
        int.all,
        features = feat,
        group.by = "sample",
        pt.size = 0
    ) +
    geom_text(
        data = medians,
        aes(x = x_pos, y = max_y * 1.05, label = sprintf("%.0f", median_value)),
        size = 3,
        color = "black",
        inherit.aes = FALSE
    ) +
    ggtitle(feat) +
    theme(legend.position = "none") +
    ylim(0, y_max)
    return(p)
})

p <- wrap_plots(plots, ncol = 2)

ggsave(
    filename = paste0(FIGS_ROOT, "/breast_cancer/suppl/QC_metrics.pdf"),
    plot = p,
    width = 8,
    height = 5
)



# pre QC metrics ALL samples and EXTERNAL datasets

#### load 


In [ ]:
qc <- readRDS(paste0(SN_RDS, "/breast_cancer/cross_patient/SIMPlex_breast_allSamples.rds"))
qc$sample <- paste0("BC_", qc$sample)
ff <- readRDS(paste0(SN_RDS, "/breast_cancer/other/patient10_seuratAnnotation.rds"))
ff$sample[ff$sample == "BC_1.1"] <- "BC_FreshFrozen_rep1"
ff$sample[ff$sample == "BC_1.2"] <- "BC_FreshFrozen_rep2"
prost1 <- readRDS(paste0(SN_RDS, "/prostate_cancer/pt10_HD_seuratAnnotation.rds"))
prost1$sample[prost1$sample == "pt10_ap5_5"] <- "Prostate_HD_pat10"
prost2 <- readRDS(paste0(SN_RDS, "/prostate_cancer/pt20_HD_seuratAnnotation.rds"))
prost2$sample[prost2$sample == "pt20_p2_2"] <- "Prostate_HD_pat20"
snPatho <- readRDS(paste0(EXT_REFS, "/snPATHO/4066_snPatho.rds"))
snPatho <- subset(snPatho, subset = sample_id == "4066FFPE") # select only FFPE sample
snPatho$sample <- snPatho$sample_id
snPatho$sample[snPatho$sample == "4066FFPE"] <- "snPatho_FFPE"
mouse <- readRDS(paste0(SN_RDS, "/mouse_brain/snRNA.rds"))
# mouse$sample[mouse$sample == "MB_2_simplex"] <- "Mouse_Brain"
tenx <- Read10X_h5(paste0(EXT_REFS, "/10x_BC_FFPE/320k_scFFPE_16-plex_GEM-X_FLEX_BreastCancer1_BC7-8_count_sample_filtered_feature_bc_matrix.h5"))
tenx <- CreateSeuratObject(counts = tenx)
tenx$sample <- "10X_FFPE_BC"

In [ ]:
## merge
qc_list <- list(qc, snPatho, tenx) # bc ffpe 
# qc_list <- list(prost1, prost2) # prostate
# qc_list <- list(qc, ff, prost1, prost2, mouse, snPatho, tenx) # all
merged <- merge(x = qc_list[[1]], y = qc_list[-1])

## ensure all nuclei > 200 unique genes
merged <- subset(merged, subset = nFeature_RNA > 200)

In [ ]:
# Define the desired sample order
sample_order <- c(
    # "10X_FFPE_BC",
    # "snPatho_FFPE",
    # # "Mouse_Brain",
    # # "BC_FreshFrozen_rep1",
    # # "BC_FreshFrozen_rep2",
    # "BC_patient1_55um",
    # "BC_patient2_55um",
    # "BC_patient4_55um",
    # "BC_patient4_HD",
    # "BC_patient5_55um",
    # "BC_patient5_HD",
    # "BC_patient6_55um",
    # "BC_patient7_55um",
    # "BC_patient8_55um",
    # "BC_patient9_55um"
     "Prostate_HD_pat10",
    "Prostate_HD_pat20"
)

# Set sample as a factor with the desired order in merged@meta.data
merged@meta.data$sample <- factor(merged@meta.data$sample, levels = sample_order)

qc_medians <- merged@meta.data %>%
    group_by(sample) %>%
    summarise(
        nFeature_RNA = median(nFeature_RNA, na.rm = TRUE),
        nCount_RNA = median(nCount_RNA, na.rm = TRUE)
    ) %>%
    tidyr::pivot_longer(
        cols = c(nFeature_RNA, nCount_RNA),
        names_to = "feature",
        values_to = "median_value"
    )

plots <- lapply(unique(qc_medians$feature), function(feat) {
    medians <- qc_medians %>% filter(feature == feat)
    # Get max value per sample for y position
    max_vals <- merged@meta.data %>%
        group_by(sample) %>%
        summarise(max_y = max(.data[[feat]], na.rm = TRUE))
    medians <- left_join(medians, max_vals, by = "sample")
    # Calculate x positions: right of each violin
    sample_levels <- levels(merged@meta.data$sample)
    n_samples <- length(sample_levels)
    x_positions <- seq_along(sample_levels)  # shift right
    medians$x_pos <- x_positions[match(medians$sample, sample_levels)]
    # Set y axis limit to 1.2x the max value
    y_max <- max(medians$max_y, na.rm = TRUE) * 1.2
    p <- VlnPlot(
        merged,
        features = feat,
        group.by = "sample",
        pt.size = 0
    ) +
    geom_text(
        data = medians,
        aes(x = x_pos, y = max_y * 1.05, label = sprintf("%.0f", median_value)),
        size = 3,
        color = "black",
        inherit.aes = FALSE
    ) +
    ggtitle(feat) +
    theme(legend.position = "none") +
    ylim(0, y_max) +
    scale_x_discrete(limits = sample_order)
    return(p)
})

p <- wrap_plots(plots, ncol = 1)

ggsave(
    filename = paste0(FIGS_ROOT, "/breast_cancer/suppl/qc_SIMPLEX_prostate.pdf"), # modify accordingly
    plot = p,
    width = 4,
    height = 12
)

In [ ]:
# Define the desired sample order
sample_order <- c(
    "10X_FFPE_BC",
    "snPatho_FFPE",
    # "Mouse_Brain",
    # "BC_FreshFrozen_rep1",
   #  "BC_FreshFrozen_rep2",
    "BC_patient1_55um",
    "BC_patient2_55um",
    "BC_patient4_55um",
    "BC_patient4_HD",
    "BC_patient5_55um",
    "BC_patient5_HD",
    "BC_patient6_55um",
    "BC_patient7_55um",
    "BC_patient8_55um",
    "BC_patient9_55um"
    # "Prostate_HD_pat10",
   #  "Prostate_HD_pat20"
)

# Ensure sample factor order in merged@meta.data
merged@meta.data$sample <- factor(merged@meta.data$sample, levels = sample_order)

# Compute cell counts per sample (ensure samples with 0 appear)
counts_tbl <- merged@meta.data %>%
    as.data.frame() %>%
    dplyr::count(sample, name = "N")

counts_df <- data.frame(sample = sample_order, stringsAsFactors = FALSE) %>%
    left_join(counts_tbl, by = "sample") %>%
    mutate(N = ifelse(is.na(N), 0, N),
                sample = factor(sample, levels = sample_order))

# Simple bar-chart (histogram-style) of cell numbers per sample
p <- ggplot(counts_df, aes(x = sample, y = N)) +
    geom_col(fill = "#6A5ACD", color = "black", width = 0.7) +
    geom_text(aes(label = N), vjust = -0.3, size = 3) +
    labs(x = "Sample", y = "Number of nuclei") +
    theme_bw() +
    theme(
        panel.grid = element_blank(),         # remove all grid lines
        panel.background = element_blank(),
        panel.border = element_blank(),       # remove box
        axis.line = element_line(color = "black"), # keep axis lines
        axis.text.x = element_text(angle = 45, hjust = 1)
    )

# Save plot
ggsave(
    filename = paste0(FIGS_ROOT, "/breast_cancer/suppl/breast_nuclei.pdf"),
    plot = p,
    width = 6,
    height = 4
)



## redo some umaps 


In [ ]:
wd <- paste0(FIGS_ROOT, "/breast_cancer/analysis_majorLevel/final_plots/")
ggsave(
    filename = paste0(wd, "celltype_umap_remake.pdf"),
    plot = DimPlot(qc, reduction = "umap", group.by = "seuratAnnotation_major", label = TRUE, alpha = 1, pt.size = 0.001, cols = colsCelltype) +
        theme(legend.position = "right"),
    width = 6,
    height = 5
)